# Automation and Affordability in U.S. Counties

Is the task composition of local work associated with what residents can afford?

Chris Bell  
Julian Pacheco

In [1]:
import os, sys
os.environ["R_HOME"]=os.path.join(sys.prefix, "lib", "R")
%load_ext rpy2.ipython
import rpy2.robjects as ro
from great_tables import style, loc
ro.r('''
suppressMessages({
  library(ggplot2)
  library(patchwork)
  library(scales)
})

# Semantic palette. Validated with the dataviz skill's validate_palette.js.
# BLUE substantive result, ORANGE failed specification, GREEN corrected or
# preferred specification, PINK flexible model, GREY context.
BLUE   <- "#2A78D6"
ORANGE <- "#EB6834"
GREEN  <- "#1BAF7A"
PINK   <- "#C2255C"
GREY   <- "#8F8D87"

# Reserve colors, available if a new semantic group needs its own. Raw hex,
# unvalidated. RED is in active use for the fig-diagnostics loess trend line.
RED     <- "#F21A00"
PURPLE  <- "#35274A"
FOREST  <- "#0B775E"
BROWN   <- "#79402E"
MAGENTA <- "#E6A0C4"

# Task groups. Scoped to these four categorical series only; lightness varies
# within each hue family so the same-family pairs stay distinguishable under
# red-green color vision deficiency. Passes validate_palette.js --pairs all at
# ΔE 17.4 (CVD) and 19.8 (normal vision). Two deliberate departures from a plain
# two-teal, two-orange scheme, both load bearing: there is no purchasing power
# color, because its continuous maps use viridis instead (a third teal shade
# failed the CVD separation check), and RM_TEAL sits toward blue rather than
# being a pure dark teal, which read as gray at low lightness in every candidate.
RC_TEAL    <- "#1E9CAD"
RM_TEAL    <- "#08599C"
NRC_ORANGE <- "#D9781F"
NRM_ORANGE <- "#8A3800"
TASK_COLORS <- c(
  "Routine Cognitive"     = RC_TEAL,
  "Routine Manual"        = RM_TEAL,
  "Non-Routine Cognitive" = NRC_ORANGE,
  "Non-Routine Manual"    = NRM_ORANGE
)

# Dollar axis labels, used by every figure reporting purchasing power.
dollar_axis <- function(v) ifelse(is.na(v), "", ifelse(v == 0, "$0",
  sprintf("%s$%s", ifelse(v < 0, "-", ""), formatC(abs(v), format="d", big.mark=","))))

theme_set(
  theme_minimal(base_size=9.5) +
    theme(
      panel.background=element_rect(fill=NA, color=NA),
      plot.background=element_rect(fill=NA, color=NA),
      panel.grid.major=element_line(color="#8F8D87", linewidth=0.3),
      panel.grid.minor=element_blank(),
      plot.title=element_text(face="bold", size=13, color="#0B0B0B", hjust=0.5, margin=margin(b=4)),
      plot.subtitle=element_text(size=8.5, color="#52514E", margin=margin(b=8)),
      strip.background=element_blank(),
      strip.text=element_text(face="bold", size=9, color="#0B0B0B"),
      axis.line=element_line(color="#0B0B0B", linewidth=0.4),
      axis.ticks=element_blank(),
      axis.text=element_text(size=9, color="#0B0B0B", face="bold"),
      axis.title=element_text(size=9.5, color="#0B0B0B"),
      legend.position="bottom",
      legend.title=element_blank(),
      legend.text=element_text(size=8.5, color="#0B0B0B"),
      legend.key=element_blank(),
      plot.margin=margin(10, 14, 8, 10)
    )
)
''')

def style_table(gt):
    """Shared table typography, applied to every GT table in the report so table
    text reads clearly smaller than body prose (great_tables defaults to 16px,
    matching body text)."""
    return gt.tab_options(
        table_font_size="12.5px",
        table_background_color="transparent",
        heading_title_font_size="14px",
        heading_subtitle_font_size="12px",
        column_labels_font_size="12.5px",
        column_labels_font_weight="bold",
        source_notes_font_size="10.5px",
    ).tab_style(
        style=style.text(style="italic"),
        locations=loc.source_notes(),
    )

R callback write-console: In addition:   
R callback write-console: Warning message:
  
R callback write-console: package ‘ggplot2’ was built under R version 4.5.3 
  

# Introduction

According to Gallup, 22 percent of American workers worried technology would make their job obsolete in 2023, up seven percentage points from 2021 ([Saad 2023](#ref-Gallup2023)). Concerns about automation were evident years earlier, when in 2017 the Pew Research Center reported that Americans expected automation to displace workers, although only 30 percent considered it likely that their own job would be replaced within their lifetime ([Smith and Anderson 2017](#ref-Pew2017)). These concerns reflect subjective expectations about future disruption rather than empirical evidence of current labor market conditions. Displacement reduces household income, and because whether that income remains sufficient depends on the local cost of living, evaluating these concerns requires an outcome that captures how households experience labor market disruption.

Median household income does not tell the full story of what residents can afford, because differences in local prices mean that the value of the same income can vary substantially across counties ([Moretti 2013](#ref-Moretti2013)). Purchasing power accounts for these differences by adjusting county median household income with Bureau of Economic Analysis Regional Price Parities. Relating that outcome to local work depends on a description of the work itself.

The framework we build on rests on three studies, beginning with Autor, Levy, and Murnane ([2003](#ref-Autor2003)), who showed that computers substitute for routine tasks and complement non-routine ones, which makes it possible to identify occupations exposed to automation. Applying that idea to local labor markets, Autor and Dorn ([2013](#ref-AutorDorn2013)) documented polarization in routine intensive areas, where employment grew at both the high and low ends of the wage distribution while middle wage employment declined. Acemoglu and Autor ([2011](#ref-AcemogluAutor2011)) then generalized the argument into a model in which tasks are the unit of production and skill groups compete to supply them.

Nevertheless, none of this work speaks directly to purchasing power at the county level. These studies report outcomes in unadjusted wages and employment counts rather than in what those wages buy locally, and they measure those outcomes at a broader labor market geography than the county. They also fit linear specifications, which leaves open whether purchasing power moves with task groups in fixed dollar steps or in proportion. These distinctions reflect differences in research purpose rather than flaws in the earlier work, since each of these studies set out to explain employment and wages rather than local affordability.

In this study we examine whether a county’s task groups are associated with purchasing power after accounting for poverty and unemployment. We use a county by year panel from 2008 to 2023, excluding 2020, with 840 counties and 11,983 observations. Every input comes from a federal source published annually at the county level, which makes the panel reproducible from public data alone. Those counties, which hold about 84 percent of the United States population ([U.S. Census Bureau 2023b](#ref-census_popest2023)), averaged 39 percent routine task content in 2023, with the middle half ranging from 35 to 43 percent. County employment is classified into four task groups, shown in <a href="#fig-taskframework" class="quarto-xref">Figure 1</a> and described in <a href="#sec-data" class="quarto-xref">Section 3</a> along with the rest of the panel construction.

In [2]:
%%R -w 8.5 -h 6 -u in -r 150 -b transparent
quad_df <- data.frame(
  group = c("Non-Routine Cognitive", "Non-Routine Manual", "Routine Cognitive", "Routine Manual"),
  col = c(2, 2, 1, 1),
  row = c(2, 1, 2, 1),
  examples = c("Management, engineering,\nteaching", "Food service, care work,\ncleaning",
               "Clerical, sales,\nclaims processing", "Production, assembly,\nmachine operation"),
  is_reference = c(TRUE, FALSE, FALSE, FALSE)
)
quad_df$group_label <- gsub(" ", "\n", quad_df$group)
quad_df$linetype <- ifelse(quad_df$is_reference, "dashed", "solid")
quad_df$linewidth <- ifelse(quad_df$is_reference, 1.3, 0.9)

box_w <- 0.42; box_h <- 0.42

col_headers <- data.frame(
  col = c(1, 2),
  label = c("Routine", "Non-Routine")
)
row_headers <- data.frame(
  row = c(2, 1),
  label = c("Cognitive", "Manual")
)

ggplot(quad_df) +
  geom_rect(aes(xmin=col-box_w, xmax=col+box_w, ymin=row-box_h, ymax=row+box_h,
                color=group, linetype=linetype, linewidth=linewidth),
            fill="#FCFCFB") +
  geom_text(aes(x=col, y=row+0.14, label=group_label, color=group),
            fontface="bold", size=4.6, lineheight=0.95) +
  geom_text(aes(x=col, y=row-0.20, label=examples), color="#52514E", size=3.1, lineheight=1.0) +
  geom_text(data=col_headers, aes(x=col, y=2.65, label=label),
            fontface="bold", size=4.3, color="#0B0B0B", inherit.aes=FALSE) +
  geom_text(data=row_headers, aes(x=0.52, y=row, label=label),
            hjust=1, size=3.6, color="#0B0B0B", lineheight=0.95, inherit.aes=FALSE) +
  scale_color_manual(values=TASK_COLORS, guide="none") +
  scale_linetype_identity() +
  scale_linewidth_identity() +
  coord_cartesian(xlim=c(0.05, 2.55), ylim=c(0.5, 2.85), clip="off") +
  theme_void() +
  theme(plot.margin=margin(10, 10, 10, 10),
        plot.background=element_rect(fill=NA, color=NA),
        plot.title=element_text(face="bold", size=13, hjust=0.5)) +
  labs(title="The Four Task Groups")

We estimate a panel regression with year indicators and standard errors clustered by county, which accounts for the dependence among repeated observations of the same county. Diagnostics on that specification motivated a log respecification, after which random forest and neural network models tested whether additional flexibility captured structure the linear specification missed. <a href="#sec-analysis" class="quarto-xref">Section 4</a> describes the modeling strategy and diagnostics.

Our primary finding is that a one percentage point shift from non-routine cognitive work to non-routine manual work is associated with roughly \$846 less in purchasing power, with a 95 percent interval of plus or minus \$38, evaluated at the panel mean. Because task groups can be measured for any county from public data, they offer information about local economic conditions that poverty and unemployment rates alone do not. That makes them useful to policymakers and economic development agencies deciding where to direct resources.

# Background

The consequences of automation differ across places because every place does not depend on the same kinds of work. Autor, Levy, and Murnane ([2003](#ref-Autor2003)) provided the framework for explaining why, arguing that rather than automating occupations wholesale, technology replaces the specific tasks within them that can be reduced to explicit rules. Routine tasks are the easiest to specify in that form, while work depending on judgment, adaptation, or direct interaction resists it. This distinction produces the four task groups used throughout this study, namely routine cognitive, routine manual, non-routine cognitive, and non-routine manual work.

Once work is understood through tasks, differences between local economies become measurable, and Autor and Dorn ([2013](#ref-AutorDorn2013)) showed that they persist across local labor markets, where they are associated with long term employment polarization. Areas with more routine work saw middle wage employment decline while employment grew at both ends of the wage distribution, a polarization traced with the occupation taxonomy this study uses to assign occupations to task groups. Acemoglu and Autor ([2011](#ref-AcemogluAutor2011)) explained why those differences matter economically, modeling tasks as the unit of production over which workers with different skills compete. Technological change shifts which workers hold an advantage, which makes the consequences of technology depend partly on the kinds of work a local economy contains.

Task groups are only one part of a county’s economic conditions, which are also described by the three conventional measures of poverty, unemployment, and median household income. Poverty identifies households with limited resources, unemployment reflects access to work, and median household income describes what households receive, but none of the three alone describes the conditions residents face.

Median household income is used because it better represents the income of a typical household than the mean, which is more sensitive to high incomes in a positively skewed distribution ([Chiripanhura 2011](#ref-Chiripanhura2011)). Because it describes what households receive rather than what they can buy, and because local prices differ, the same nominal income supports different living standards in different places. Purchasing power combines the two by adjusting income with Regional Price Parities, which can change how counties compare rather than shifting them all by the same amount.

<a href="#fig-county-context" class="quarto-xref">Figure 2</a> maps those three measures across counties, together with the Regional Price Parities used to adjust income. Poverty (Panel A) and unemployment (Panel B) mark different concentrations of distress. Median household income (Panel C) shows where resources are highest and lowest before prices are taken into account, while Regional Price Parity (Panel D) shows where those prices are high or low. The patterns do not align. A county that looks strong under one measure can look weaker under another, which is why no single conventional measure summarizes local economic conditions.

In [3]:
import os
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine

engine = create_engine(
    os.environ["AUTORACK_URL"],
    pool_pre_ping=True,
    pool_recycle=300
)

In [4]:
context_year = 2023
context_df = pd.read_sql("""
    select cb.county_fips, cb.poverty_rate, cb.unemployment_rate,
           cb.median_household_income, ca.affordability_salary as purchasing_power
    from county_baseline cb
    join county_affordability ca
      on ca.county_fips = cb.county_fips and ca.year = cb.year
    where cb.year = %(yr)s
""", engine, params={"yr": context_year})
context_df["fips"] = context_df["county_fips"].astype(str).str.zfill(5)
# RPP recovered from the already computed purchasing power measure (@eq-afford, inverted):
# purchasing power = income / (RPP / 100), so RPP = income / purchasing power * 100.
context_df["rpp"] = context_df["median_household_income"] / context_df["purchasing_power"] * 100

gdf_context = gpd.read_file(
    "https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip"
)
gdf_context = gdf_context.rename(columns={"GEOID": "fips"})
merged_context = gdf_context.merge(
    context_df[["fips", "poverty_rate", "unemployment_rate",
                "median_household_income", "rpp"]],
    on="fips", how="left"
)
merged_context = merged_context[
    ~merged_context["STATEFP"].isin(["02", "15", "60", "66", "69", "72", "78"])
].copy()

pov_ctx_vmin = float(context_df["poverty_rate"].quantile(0.02))
pov_ctx_vmax = float(context_df["poverty_rate"].quantile(0.98))
unemp_ctx_vmin = float(context_df["unemployment_rate"].quantile(0.02))
unemp_ctx_vmax = float(context_df["unemployment_rate"].quantile(0.98))
inc_ctx_vmin = float(context_df["median_household_income"].quantile(0.02))
inc_ctx_vmax = float(context_df["median_household_income"].quantile(0.98))
rpp_ctx_vmin = float(context_df["rpp"].quantile(0.02))
rpp_ctx_vmax = float(context_df["rpp"].quantile(0.98))

os.makedirs("output", exist_ok=True)
merged_context[["fips", "poverty_rate", "unemployment_rate",
                "median_household_income", "rpp", "geometry"]].to_file(
    "output/county_context.geojson", driver="GeoJSON"
)

In [5]:
%%R -i pov_ctx_vmin -i pov_ctx_vmax -i unemp_ctx_vmin -i unemp_ctx_vmax -i inc_ctx_vmin -i inc_ctx_vmax -i rpp_ctx_vmin -i rpp_ctx_vmax -w 10 -h 9 -u in -r 150 -b transparent
suppressMessages(library(sf))

context_sf <- st_read("output/county_context.geojson", quiet=TRUE)

context_panel_theme <- theme_void() +
  theme(legend.position="bottom", legend.key.width=unit(1.1, "cm"),
        plot.title=element_text(face="bold", size=11, hjust=0.5),
        plot.background=element_rect(fill=NA, color=NA))

bottom_guide <- guide_colorbar(direction="horizontal", title.position="bottom", title.hjust=0.5)

p_pov <- ggplot(context_sf) +
  geom_sf(aes(fill=poverty_rate), color="white", linewidth=0.05) +
  scale_fill_gradient(low="#FCFCFB", high=ORANGE, limits=c(pov_ctx_vmin, pov_ctx_vmax),
                      oob=scales::squish, na.value="#EEEEEE", name="Poverty Rate (%)",
                      guide=bottom_guide) +
  coord_sf(crs=st_crs(5070), datum=NA) +
  context_panel_theme +
  labs(title="Poverty Rate")

p_unemp <- ggplot(context_sf) +
  geom_sf(aes(fill=unemployment_rate), color="white", linewidth=0.05) +
  scale_fill_gradient(low="#FCFCFB", high=PINK, limits=c(unemp_ctx_vmin, unemp_ctx_vmax),
                      oob=scales::squish, na.value="#EEEEEE", name="Unemployment Rate (%)",
                      guide=bottom_guide) +
  coord_sf(crs=st_crs(5070), datum=NA) +
  context_panel_theme +
  labs(title="Unemployment Rate")

p_inc <- ggplot(context_sf) +
  geom_sf(aes(fill=median_household_income), color="white", linewidth=0.05) +
  scale_fill_gradient(low="#FCFCFB", high=GREEN, limits=c(inc_ctx_vmin, inc_ctx_vmax),
                      oob=scales::squish, na.value="#EEEEEE",
                      labels=function(v) sprintf("$%.0fk", v/1000),
                      name="Median Household Income",
                      guide=bottom_guide) +
  coord_sf(crs=st_crs(5070), datum=NA) +
  context_panel_theme +
  labs(title="Median Household Income")

p_rpp <- ggplot(context_sf) +
  geom_sf(aes(fill=rpp), color="white", linewidth=0.05) +
  scale_fill_gradient(low="#FCFCFB", high=BLUE, limits=c(rpp_ctx_vmin, rpp_ctx_vmax),
                      oob=scales::squish, na.value="#EEEEEE",
                      name="Regional Price Parity",
                      guide=bottom_guide) +
  coord_sf(crs=st_crs(5070), datum=NA) +
  context_panel_theme +
  labs(title="Regional Price Parity")

(p_pov | p_unemp) / (p_inc | p_rpp) +
  plot_annotation(title="Four Views of County Economic Conditions",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B")))

The geography used to measure these conditions also matters, since much of the previous literature examines broader labor market areas that group several counties together. That scale suits the study of employment adjustment but can combine counties with different local prices into a single market, since housing and other costs vary substantially between a metropolitan core and its surrounding counties. A county level design preserves that variation.

One final consideration is how the relationship between task groups and purchasing power is represented. A fixed dollar specification assumes the same shift in task groups corresponds to the same dollar difference everywhere. A proportional one instead implies a larger dollar difference where purchasing power is already high and a smaller one where it is low, a distinction the analysis evaluates empirically rather than assuming from the outset.

Together these considerations define the gap this study addresses. Prior research shows why task groups matter for local labor markets, and conventional measures describe other dimensions of county conditions, but neither connects the two to what income actually buys. That connection is also missing at the county scale, which preserves the price differences a broader labor market area averages out. The form of that relationship is untested as well, since no prior study establishes whether it runs in fixed dollars or in proportion.

# Data

## Data sources and study period

This study combines five federal datasets published by three agencies, all retrieved in June 2026 and summarized in <a href="#tbl-sources" class="quarto-xref">Table 1</a> with the number of records and the variables each source supplies.

In [6]:
import pandas as pd
from great_tables import GT

sources_df=pd.DataFrame({
    "Data Source": ["Small Area Income and Poverty Estimates<br>**Census SAIPE**",
                    "Bureau of Economic Analysis Regional Price Parities<br>**BEA RPP**",
                    "Core Based Statistical Area delineation<br>**Census CBSA**",
                    "American Community Survey 1 year estimates<br>**Census ACS**",
                    "Bureau of Labor Statistics Local Area Unemployment Statistics<br>**BLS LAUS**"],
    "Records": [50283, 884, 387, 15690, 88004],
    "Measures": ["median household income, poverty rate",
                 "county price level relative to the national average",
                 "county to metropolitan area crosswalk",
                 "population, occupational employment by category",
                 "unemployment rate"],
    "Destination Table": ["`county_baseline`","`cbsa_rpp`","`cbsa`",
                          "`county_baseline`, `county_task_exposure`","`county_baseline`"],
})

style_table(GT(sources_df)
  .fmt_integer(columns="Records", use_seps=True)
  .fmt_markdown(columns=["Data Source","Destination Table"])
  .cols_align(align="center", columns=["Data Source","Records","Measures","Destination Table"])
  .tab_source_note("Scale: 3,143 counties per year, 15 years (2008 to 2023, excluding 2020), 47,140 county year universe, 11,983 rows in the analytical panel.")
  .tab_source_note("Records count rows retrieved from each source rather than analytical observations. The ACS publishes 1 year estimates only for areas above 65,000 residents, which reduces the universe to the 848 county panel.")
  .tab_source_note("All sources retrieved June 2026 through agency APIs or bulk file download."))

Most sources report at the county year level, joined on county Federal Information Processing Standards (FIPS) code and year. Regional Price Parities (RPP) are published at the metropolitan area level instead, which requires an additional step. Counties are first linked to their metropolitan area using the Census Core Based Statistical Area (CBSA) delineation file, then matched to the corresponding Bureau of Economic Analysis (BEA) price measure ([U.S. Bureau of Economic Analysis 2024](#ref-bea_rpp); [U.S. Census Bureau 2023a](#ref-census_cbsa)).

The study spans 2008 through 2023 but excludes 2020, because the American Community Survey (ACS) did not publish its standard 1 year estimates that year after disruptions to data collection during the COVID-19 pandemic ([U.S. Census Bureau 2024](#ref-census_acs)). Those estimates provide the occupational employment counts used to construct the task groups. Across the remaining fifteen years, the Small Area Income and Poverty Estimates (SAIPE) program provides an initial sample of 47,140 county year observations.[1]

## Analytical sample

Not all observations contain the occupational data required for estimation, and the first restriction on the sample therefore comes from the coverage of the ACS occupational estimates. The Census Bureau publishes ACS 1 year occupational estimates only for areas with populations of at least 65,000. Counties below this threshold lack the employment counts needed to construct the four task groups. Applying this restriction reduces the sample to 12,087 county year observations across 848 counties.

The second restriction requires complete values for the model covariates, and it removes an additional 104 observations for missing unemployment rates, while population, median household income, and poverty rate are complete throughout.

All 104 missing observations belong to Connecticut’s eight legacy counties, for which the Bureau of Labor Statistics does not report unemployment rates at the geography used elsewhere in the panel ([U.S. Bureau of Labor Statistics 2024](#ref-bls_laus)). The missingness reflects county geography rather than the observed unemployment rate itself, which makes it missing at random rather than missing completely at random.

The final analytical sample contains 11,983 county year observations across 840 of the 848 counties, since Connecticut’s eight legacy counties lose every year and drop out entirely. All models reported in <a href="#sec-analysis" class="quarto-xref">Section 4</a> and <a href="#sec-results" class="quarto-xref">Section 5</a> are estimated using these same observations.

For machine learning evaluation, the data are partitioned at the county level rather than by individual county year observations. Because each county can appear in the panel for up to fifteen years, a random row level split could place observations from the same county in both the training and test sets. Grouping the split by county prevents this overlap and ensures that all observations from a county remain entirely within one partition. <a href="#sec-analysis" class="quarto-xref">Section 4</a> describes the evaluation procedure in detail.

The population threshold also limits the scope of the results, since the 840 retained counties contain about 84 percent of the United States population but represent a minority of all counties. Rural counties are therefore underrepresented, because the retained counties are generally more populous and metropolitan than those the ACS threshold excludes. The regression and machine learning results describe counties above the ACS publication threshold rather than counties nationwide, a limitation <a href="#sec-conclusions" class="quarto-xref">Section 6</a> returns to.

## Construction of task group measures

Following the task framework introduced in <a href="#sec-background" class="quarto-xref">Section 2</a>, we grouped the broad occupational categories reported in the ACS 1 year estimates into four task groups. They are routine cognitive, routine manual, non-routine cognitive, and non-routine manual. Because the Census categories do not carry these labels directly, we applied the classification logic of Autor and Dorn ([2013](#ref-AutorDorn2013)). Clerical and sales occupations were classified as routine cognitive; production and construction occupations as routine manual; managerial, professional, and technical occupations as non-routine cognitive; and service occupations as non-routine manual. For each county and year, employment was then summed across the occupational categories assigned to each group, as shown in <a href="#eq-totals" class="quarto-xref">Equation 1</a>.

<span id="eq-totals">$$
T_{g,c,t} = \sum_{o \in g} E_{o,c,t}
 \qquad(1)$$</span>

Here $E_{o,c,t}$ represents employment in occupational category $o$ for county $c$ during year $t$, and the sum includes all occupations assigned to task group $g$. Because the calculation uses employment counts only and applies no task intensity weights, each group measures the amount of county employment associated with that type of work rather than its intensity. Dividing each group total by total employment, as shown in <a href="#eq-group" class="quarto-xref">Equation 2</a>, converts the four values to proportions between zero and one that sum to one. Together, these proportions describe how a county’s employment splits across the four task groups.

<span id="eq-group">$$
G_{g,c,t} = \frac{T_{g,c,t}}{\sum_{g'} T_{g',c,t}}
 \qquad(2)$$</span>

These proportions place counties of different sizes on a common scale, allowing the analysis to compare task groups rather than the size of the workforce. Because the four proportions sum to one, one group must be omitted from the regression to avoid perfect collinearity. Non-routine cognitive work is the reference group, and the remaining coefficients are interpreted relative to it. <a href="#sec-analysis" class="quarto-xref">Section 4</a> explains how these task groups are incorporated into the models.

We retain all four task groups in the data rather than combining them into a single routine intensity index because the type of work matters. Under a single index, a county shifting away from clerical work and a county shifting away from assembly work could appear identical even though the underlying changes in task groups are different.

Comparisons over time are also complicated by a change in the source data, since the Census Bureau revised its occupational classification between 2009 and 2010, changing some category boundaries while preserving their assignment to the four task groups. <a href="#sec-analysis" class="quarto-xref">Section 4</a> examines whether this classification change is visible in the observed variation and reports the corresponding robustness checks.

## Construction of the purchasing power measure

To account for differences in what household income can buy across counties, the outcome is measured as local purchasing power. Regional Price Parities measure differences in price levels across places relative to the national price level and allow comparisons of buying power across regions ([U.S. Bureau of Economic Analysis 2024](#ref-bea_rpp)). We apply this adjustment to median household income from Census SAIPE. Because BEA expresses RPP as a percentage of the national price level, dividing the index by 100 converts it to a price multiplier. Median household income is then divided by this multiplier to obtain the purchasing power measure defined in <a href="#eq-afford" class="quarto-xref">Equation 3</a>.

<span id="eq-afford">$$
A_{c,t} = \frac{\text{Median household income}_{c,t}}{\text{RPP}_{c,t} / 100}
 \qquad(3)$$</span>

A county with a median household income of \$60,000 and a Regional Price Parity of 120 has purchasing power of \$50,000. In practical terms, that income buys roughly what \$50,000 would buy at national average prices. This adjustment makes household income more comparable across counties by accounting for differences in local prices.

Because BEA does not publish a separate Regional Price Parity for counties outside metropolitan areas, those counties are assigned the corresponding state level parity, which extends price coverage to the full sample. Within the analytical panel, 45.5 percent of county year observations use a metropolitan parity and 54.5 percent use the state measure. For more than half the panel the price adjustment is therefore a state average, so variation in prices within a state is not captured, and counties assigned the same state value may face quite different local costs.

### Inflation and year indicators

Regional Price Parities adjust for differences across places but not for changes in the national price level over time, so purchasing power is reported in nominal U.S. dollars. The familiar alternative is to deflate those dollars to a constant base year with a national price index, but the regression uses year indicators, which absorb the same annual constant.

A national deflator is a price index, such as the Consumer Price Index, that restates dollars from different years in the prices of a single base year. Because it takes one value per year and applies that value to every county, what it does to the estimates depends on the specification, and the panel regression used for inference models the logarithm of purchasing power. In logs, dividing by the deflator becomes subtracting the same amount from every observation in a year, which is exactly the kind of constant a year indicator absorbs. Deflating would therefore move the year coefficients and leave every other coefficient where it was, including the task groups. The indicators also absorb anything shared across counties within a year that a price index does not measure, such as common movements in incomes. The cost is that the year coefficients cannot be read as a measure of inflation, since they carry price drift and every other common annual movement together. <a href="#sec-analysis" class="quarto-xref">Section 4</a> explains how these year indicators enter the model, and <a href="#sec-conclusions" class="quarto-xref">Section 6</a> returns to this limitation.

## Control variables

The models include three county level covariates to account for economic conditions associated with both task groups and purchasing power. Poverty rate and unemployment rate are the conventional measures of county economic distress introduced in <a href="#sec-background" class="quarto-xref">Section 2</a>, and including them tests whether task groups carry information those measures do not already contain. Median household income is the third conventional measure, but it enters the outcome directly and cannot also serve as a control. Population is included for a different reason, since county scale varies by orders of magnitude and the estimates would otherwise partly reflect how large a county is rather than the kind of work it contains.

<a href="#fig-baseline-dists" class="quarto-xref">Figure 3</a> shows the distribution of these three covariates, together with median household income, across the same task group panel <a href="#fig-dists" class="quarto-xref">Figure 9</a> uses for purchasing power and the task groups. Population is by far the most skewed of the four, which is why it enters the models in logarithmic form rather than its raw scale. Poverty rate, unemployment rate, and median household income are all right skewed but far less extreme, consistent with the moderate skew already documented for purchasing power itself.

[1] The District of Columbia and Kalawao County, Hawaii, are excluded because they are absent from the county reference file. Connecticut counties leave the panel after 2021 because the state replaced its legacy counties with planning regions beginning in 2022.

In [7]:
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
baseline_engine=create_engine(os.environ["AUTORACK_URL"], pool_pre_ping=True, pool_recycle=300)
baseline_df=pd.read_sql("""
select ca.county_fips, ca.year, cb.poverty_rate, cb.unemployment_rate,
    cb.population, cb.median_household_income
from county_affordability ca
join county_task_exposure cte on ca.county_fips=cte.county_fips and ca.year=cte.year
join county_baseline cb on ca.county_fips=cb.county_fips and ca.year=cb.year
""", baseline_engine)

baseline_names={"poverty_rate":"Poverty Rate (%)", "unemployment_rate":"Unemployment Rate (%)",
                "population":"Population", "median_household_income":"Median Household Income ($)"}
baseline_long=baseline_df.melt(value_vars=list(baseline_names.keys()),
                               var_name="variable", value_name="value")
baseline_long["variable"]=baseline_long["variable"].map(baseline_names)

In [8]:
%%R -i baseline_long -w 9 -h 7 -u in -r 150 -b transparent
baseline_order <- c("Poverty Rate (%)","Unemployment Rate (%)","Population","Median Household Income ($)")
baseline_long$variable <- factor(baseline_long$variable, levels=baseline_order)

p_pov <- ggplot(subset(baseline_long, variable=="Poverty Rate (%)"), aes(x=variable, y=value)) +
  geom_violin(fill=ORANGE, color=ORANGE, alpha=0.35, linewidth=0.6, trim=TRUE) +
  geom_boxplot(width=0.12, fill="white", color="grey30", alpha=0.7,
               outlier.size=0.8, outlier.alpha=0.5) +
  scale_y_continuous(labels=function(v) sprintf("%g%%", v), breaks=scales::pretty_breaks(n=4)) +
  labs(title="Poverty Rate", x=NULL, y="Rate") +
  theme(axis.text.x=element_blank(), plot.title=element_text(size=11),
        panel.grid.major=element_blank(), panel.grid.minor=element_blank())

p_unemp <- ggplot(subset(baseline_long, variable=="Unemployment Rate (%)"), aes(x=variable, y=value)) +
  geom_violin(fill=PINK, color=PINK, alpha=0.35, linewidth=0.6, trim=TRUE) +
  geom_boxplot(width=0.12, fill="white", color="grey30", alpha=0.7,
               outlier.size=0.8, outlier.alpha=0.5) +
  scale_y_continuous(labels=function(v) sprintf("%g%%", v), breaks=scales::pretty_breaks(n=4)) +
  labs(title="Unemployment Rate", x=NULL, y="Rate") +
  theme(axis.text.x=element_blank(), plot.title=element_text(size=11),
        panel.grid.major=element_blank(), panel.grid.minor=element_blank())

p_pop <- ggplot(subset(baseline_long, variable=="Population"), aes(x=variable, y=value)) +
  geom_violin(fill=GREY, color=GREY, alpha=0.35, linewidth=0.6, trim=TRUE) +
  geom_boxplot(width=0.12, fill="white", color="grey30", alpha=0.7,
               outlier.size=0.8, outlier.alpha=0.5) +
  scale_y_log10(labels=label_comma(), breaks=scales::log_breaks(n=4)) +
  labs(title="Population", x=NULL, y="Population") +
  theme(axis.text.x=element_blank(), plot.title=element_text(size=11),
        panel.grid.major=element_blank(), panel.grid.minor=element_blank())

p_inc <- ggplot(subset(baseline_long, variable=="Median Household Income ($)"), aes(x=variable, y=value)) +
  geom_violin(fill=GREEN, color=GREEN, alpha=0.35, linewidth=0.6, trim=TRUE) +
  geom_boxplot(width=0.12, fill="white", color="grey30", alpha=0.7,
               outlier.size=0.8, outlier.alpha=0.5) +
  scale_y_continuous(labels=dollar_axis, breaks=scales::pretty_breaks(n=4)) +
  labs(title="Median Household Income", x=NULL, y="Income") +
  theme(axis.text.x=element_blank(), plot.title=element_text(size=11),
        panel.grid.major=element_blank(), panel.grid.minor=element_blank())

(p_pov | p_unemp) / (p_pop | p_inc) +
  plot_annotation(title="Distribution of the Control Variables",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B")))

R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1: Removed 104 rows containing non-finite outside the scale range
(`stat_ydensity()`). 
  
R callback write-console: 2: Removed 104 rows containing non-finite outside the scale range
(`stat_boxplot()`). 
  

Every covariate here is right skewed to some degree, but only population is skewed enough to change how it enters the models. Poverty rate, unemployment rate, and median household income are used in their raw form throughout. That is consistent with the log respecification in <a href="#sec-analysis" class="quarto-xref">Section 4</a>, which rests on the outcome’s own skew rather than a blanket rule applied to every variable.

## Data architecture and organization

The data architecture separates raw source records from the tables used for analysis. Raw retrievals and intermediate results are stored in a PostgreSQL data lake containing 41 tables. These records are intentionally left unnormalized to preserve traceability and allow each processing step to be reproduced without returning to the original agency source.

Processed data are then written to an analytical warehouse containing eleven tables, with the core analytical tables organized in third normal form (3NF). Before entering the warehouse, county identifiers are standardized, records are restricted to the study period, jurisdictions outside the panel are removed, and foreign key constraints are enforced. This creates a consistent structure in which every analytical record can be linked to a valid county.

<a href="#fig-erd" class="quarto-xref">Figure 4</a> summarizes the full path from federal sources through the data lake and warehouse to the analysis ready tables. The figure emphasizes the eight warehouse tables used directly in the analysis and omits three supporting tables that are not required for the models reported here.

In [9]:
import graphviz

def entity(name, rows):
    body = f'<TR><TD COLSPAN="3" BGCOLOR="#2A78D6"><FONT COLOR="white" POINT-SIZE="15"><B>{name}</B></FONT></TD></TR>'
    for typ, col, marker in rows:
        m = f'<FONT COLOR="#8F8D87" POINT-SIZE="10">{marker}</FONT>' if marker else ""
        body += f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="10">{typ}</FONT></TD><TD ALIGN="LEFT"><FONT POINT-SIZE="13">{col}</FONT></TD><TD ALIGN="LEFT">{m}</TD></TR>'
    return f'<<TABLE BORDER="1" CELLBORDER="0" CELLSPACING="0" CELLPADDING="3">{body}</TABLE>>'

CENSUS_FILL = "#DCEEF7"; CENSUS_BORDER = "#2A78D6"
BEA_FILL = "#DCF3EA"; BEA_BORDER = "#1BAF7A"
BLS_FILL = "#F7DCE6"; BLS_BORDER = "#C2255C"
OUT_FILL = "#FBE3D3"; OUT_BORDER = "#EB6834"
LAKE_FILL = "#EDEDEA"; LAKE_BORDER = "#8F8D87"

dot = f'''
digraph architecture {{
  rankdir=TB
  compound=true
  bgcolor="transparent"
  fontname="Helvetica"
  label=<<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0"><TR><TD><FONT FACE="Helvetica-Bold" POINT-SIZE="24">Data Pipeline Architecture</FONT></TD></TR></TABLE>>
  labelloc="t"
  nodesep=0.4
  ranksep=0.9
  node [fontname="Helvetica", shape=box, style="rounded", color="#C3C2B7"]
  edge [fontname="Helvetica", color="black", arrowsize=0.6]

  subgraph cluster_source {{
    label="Data Sources"
    labeljust="c"
    fontsize=16
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#F2F8FC"
    margin=18
    node [style="filled,rounded", margin="0.1,0.05", fontsize=14]
    src_census [label="Census Bureau\\nSAIPE · ACS 1yr · CBSA delineation", fillcolor="{CENSUS_FILL}", color="{CENSUS_BORDER}"]
    src_bea [label="BEA\\nRegional Price Parities", fillcolor="{BEA_FILL}", color="{BEA_BORDER}"]
    src_bls [label="BLS\\nLAUS", fillcolor="{BLS_FILL}", color="{BLS_BORDER}"]
  }}

  subgraph cluster_lake {{
    label="Data Lake"
    labeljust="c"
    fontsize=16
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#F4F4F1"
    margin=18
    node [style="filled,rounded", fillcolor="{LAKE_FILL}", color="{LAKE_BORDER}", margin="0.1,0.05", fontsize=14]
    lake [label="Raw retrievals and intermediate results\\n41 tables, unnormalized"]
  }}

  subgraph cluster_warehouse {{
    label="Data Warehouse (3NF)"
    labeljust="c"
    fontsize=16
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#F2FAF6"
    margin=18
    node [shape=plain]

    state [label={entity("state", [("varchar","state_code","PK"),("varchar","state_name","")])}]
    county [label={entity("county", [("bigint","county_fips","PK"),("varchar","county_name",""),("varchar","state_code","FK")])}]
    county_baseline [label={entity("county_baseline", [("bigint","county_fips","PK,FK"),("int","year","PK"),("numeric","population",""),("numeric","median_household_income",""),("numeric","poverty_rate",""),("numeric","unemployment_rate","")])}]
    county_task_exposure [label={entity("county_task_exposure", [("bigint","county_fips","PK,FK"),("int","year","PK"),("numeric","routine_cognitive_share","generated"),("numeric","routine_manual_share","generated"),("numeric","non_routine_cognitive_share","generated"),("numeric","non_routine_manual_share","generated")])}]
    county_affordability [label={entity("county_affordability", [("bigint","county_fips","PK,FK"),("int","year","PK"),("numeric","affordability_salary","")])}]
    county_cbsa_code [label={entity("county_cbsa_code", [("bigint","county_fips","PK,FK"),("varchar","cbsa_code","PK,FK")])}]
    cbsa [label={entity("cbsa", [("varchar","cbsa_code","PK"),("varchar","cbsa_name","")])}]
    cbsa_rpp [label={entity("cbsa_rpp", [("varchar","cbsa_code","PK,FK"),("int","year","PK"),("numeric","rpp_value","")])}]

    {{rank=same; county_baseline; county_task_exposure; county_affordability}}

    state -> county
    county -> county_baseline
    county -> county_task_exposure
    county -> county_affordability
    county -> county_cbsa_code
    county_cbsa_code -> cbsa
    cbsa -> cbsa_rpp
  }}

  subgraph cluster_output {{
    label="Modeling Tables"
    labeljust="c"
    fontsize=16
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#FDF5EF"
    margin=18
    node [style="filled,rounded", fillcolor="{OUT_FILL}", color="{OUT_BORDER}", margin="0.1,0.05", fontsize=14]
    analysis_table [label="Analysis ready county year table\\n(joined on county_fips and year)"]
    models [label="Statistical and ML models"]
    figures [label="Figures and tables"]
    {{rank=same; models; figures}}
    analysis_table -> models
    analysis_table -> figures
  }}

  src_census -> lake [ltail="cluster_source", lhead="cluster_lake", minlen=2]
  src_bea -> lake [ltail="cluster_source", lhead="cluster_lake", minlen=2]
  src_bls -> lake [ltail="cluster_source", lhead="cluster_lake", minlen=2]
  lake -> state [ltail="cluster_lake", lhead="cluster_warehouse", minlen=2]
  county_affordability -> analysis_table [ltail="cluster_warehouse", lhead="cluster_output", minlen=2]
  county_baseline -> analysis_table [style=invis]
  county_task_exposure -> analysis_table [style=invis]
}}
'''
graphviz.Source(dot)

The warehouse is organized around the county table, which is keyed by FIPS code and linked to state as a reference table. Three analytical tables join to each county by FIPS code and year. county_baseline contains population, median household income, poverty rate, and unemployment rate. county_task_exposure contains the four task group totals defined in <a href="#eq-totals" class="quarto-xref">Equation 1</a>, while county_affordability contains the purchasing power outcome.

The normalized task proportions defined in <a href="#eq-group" class="quarto-xref">Equation 2</a> are stored as generated columns, allowing the database to derive them directly from the four task totals and maintain a consistent calculation across the analysis. Regional Price Parities are kept on their own relational path for the same reason, since copying each metropolitan parity onto every county in that area would duplicate the same value across records.

# Analysis

In [10]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

engine=create_engine(os.environ["AUTORACK_URL"], pool_pre_ping=True, pool_recycle=300)
df=pd.read_sql("""
select ca.county_fips, ca.year, ca.affordability_salary,
    cte.routine_cognitive_share, cte.routine_manual_share,
    cte.non_routine_cognitive_share, cte.non_routine_manual_share,
    cb.poverty_rate, cb.unemployment_rate, cb.population
from county_affordability ca
join county_task_exposure cte on ca.county_fips=cte.county_fips and ca.year=cte.year
join county_baseline cb on ca.county_fips=cb.county_fips and ca.year=cb.year
""", engine)
df["log_population"]=np.log(df["population"])

In [11]:
# level model: OLS with year indicators, standard errors clustered by county
reg_cols=["routine_cognitive_share","routine_manual_share","non_routine_manual_share",
          "poverty_rate","unemployment_rate","log_population"]

reg=df.dropna(subset=reg_cols+["affordability_salary"]).copy()

X_panel=pd.concat([reg[reg_cols],
                   pd.get_dummies(reg["year"], prefix="year", drop_first=True).astype(float)], axis=1)
X_panel=sm.add_constant(X_panel)

model=sm.OLS(reg["affordability_salary"], X_panel).fit(
    cov_type="cluster", cov_kwds={"groups": reg["county_fips"]})

reg["fitted"]=model.fittedvalues
reg["resid"]=model.resid

# log respecification, same design matrix
reg["actual_log"]=np.log(reg["affordability_salary"])
model_log=sm.OLS(reg["actual_log"], X_panel).fit(
    cov_type="cluster", cov_kwds={"groups": reg["county_fips"]})
reg["resid_log"]=model_log.resid
reg["fitted_log"]=model_log.fittedvalues

## The unadjusted relationship

The ladder starts with the relationship itself, before any controls. <a href="#fig-univariate" class="quarto-xref">Figure 5</a> plots purchasing power against each of the four task groups across the panel. The two manual groups slope down most steeply, non-routine cognitive slopes up, and routine cognitive is the flattest of the four. None of the four is tight enough to carry a claim on its own, which is why the models that follow adjust for the rest of the panel. The ordering survives that adjustment.

In [12]:
uni_labels={
    "routine_cognitive_share":"Routine Cognitive",
    "routine_manual_share":"Routine Manual",
    "non_routine_cognitive_share":"Non-Routine Cognitive",
    "non_routine_manual_share":"Non-Routine Manual",
}
uni_pts=df.melt(id_vars="affordability_salary", value_vars=list(uni_labels.keys()),
                var_name="group", value_name="share").dropna()
uni_pts["group"]=uni_pts["group"].map(uni_labels)
uni_pts["share"]=uni_pts["share"]*100  # express as percent of county employment

uni_pts["bin"]=uni_pts.groupby("group")["share"].transform(
    lambda s: pd.qcut(s, 20, labels=False, duplicates="drop"))
uni_bins=(uni_pts.groupby(["group","bin"])[["share","affordability_salary"]]
          .mean().reset_index())
uni_order=list(uni_labels.values())

In [13]:
%%R -i uni_pts -i uni_bins -i uni_order -w 10 -h 6 -u in -r 150 -b transparent
uni_pts$group <- factor(uni_pts$group, levels=unlist(uni_order))
uni_bins$group <- factor(uni_bins$group, levels=unlist(uni_order))

# the strip carries the group name alone, centered; the letter tag sits just above
# the top of each panel's y axis, clear of both the strip and the axis labels
tag_df <- data.frame(group=factor(unlist(uni_order), levels=unlist(uni_order)),
                     tag=paste0(LETTERS[seq_along(unlist(uni_order))], ")"))

ggplot(uni_pts, aes(x=share, y=affordability_salary)) +
  geom_point(alpha=0.08, size=0.25, color="grey40") +
  geom_line(data=uni_bins, aes(color=group), linewidth=0.8, show.legend=FALSE) +
  geom_point(data=uni_bins, aes(color=group), size=1.8, show.legend=FALSE) +
  geom_text(data=tag_df, aes(x=-Inf, y=Inf, label=tag), inherit.aes=FALSE,
            hjust=1.2, vjust=-0.45, fontface="bold", size=4.2, color="#0B0B0B") +
  # axes="all_y" repeats the y axis on the right column too; the scale stays shared
  facet_wrap(~group, ncol=2, scales="free_x", axes="all_y") +
  coord_cartesian(clip="off") +
  scale_color_manual(values=TASK_COLORS) +
  scale_x_continuous(breaks=scales::pretty_breaks(n=8)) +
  scale_y_continuous(labels=dollar_axis) +
  labs(title="Purchasing Power Against Each Task Group",
       x="Percent of County Employment", y="Purchasing Power") +
  theme(plot.title=element_text(hjust=0.5),
        strip.text=element_text(size=12, hjust=0.5),
        axis.ticks.x=element_line(color="#C3C2B7", linewidth=0.3),
        panel.grid.major=element_blank(), panel.grid.minor=element_blank(),
        panel.spacing=unit(1.4, "lines"))

## Variance decomposition

Purchasing power and the task groups vary both across counties and within a county over time, and the two carry different implications. <a href="#fig-between-within" class="quarto-xref">Figure 6</a> splits each group’s variation into the two components. The manual groups vary almost entirely between counties, which is what makes the purchasing power differences they carry durable rather than temporary. Routine cognitive is the only group with substantial within county movement.

In [14]:
df10=df[df["year"]>=2010].copy()
input_cols=["routine_cognitive_share","routine_manual_share","non_routine_manual_share"]

within_vals=[]
for col in input_cols:
    yd=df10[col]-df10.groupby("year")[col].transform("mean")
    within=(yd-yd.groupby(df10["county_fips"]).transform("mean")).var()
    within_vals.append(within/yd.var()*100)

bw_labels=["Routine Cognitive","Routine Manual","Non-Routine Manual"]
bw_df=pd.concat([
    pd.DataFrame({"group":bw_labels, "value":[100-v for v in within_vals],
                  "component":"Between Counties"}),
    pd.DataFrame({"group":bw_labels, "value":within_vals,
                  "component":"Within Counties Over Time"}),
], ignore_index=True)
bw_text_df=pd.concat([
    pd.DataFrame({"group":bw_labels, "x":[(100-v)/2 for v in within_vals],
                  "label":[f"{100-v:.0f}%" for v in within_vals], "component":"Between Counties"}),
    pd.DataFrame({"group":bw_labels, "x":[100-v/2 for v in within_vals],
                  "label":[f"{v:.0f}%" for v in within_vals], "component":"Within Counties Over Time"}),
], ignore_index=True)

In [15]:
%%R -i bw_df -i bw_text_df -w 9 -h 3.5 -u in -r 150 -b transparent
bw_df$group <- factor(bw_df$group,
    levels=rev(c("Routine Cognitive","Routine Manual","Non-Routine Manual")))
bw_df$component <- factor(bw_df$component,
    levels=c("Within Counties Over Time","Between Counties"))
bw_text_df$group <- factor(bw_text_df$group, levels=levels(bw_df$group))

# Orange and purple rather than green and red: these are two components of one
# variance, neither good nor bad, so a valenced pair would say the wrong thing.
# Both in-bar labels clear the 4.5:1 floor (orange with ink 6.2, purple with white 13.6).
bw_text_df$text_color <- ifelse(bw_text_df$component=="Within Counties Over Time",
                                 "white", "#0B0B0B")

ggplot(bw_df, aes(x=value, y=group, fill=component)) +
  geom_col(width=0.7) +
  geom_text(data=bw_text_df, aes(x=x, y=group, label=label, color=text_color),
            inherit.aes=FALSE, size=3.4) +
  scale_color_identity() +
  scale_fill_manual(values=c("Between Counties"=ORANGE,
                             "Within Counties Over Time"=PURPLE), name=NULL) +
  scale_x_continuous(breaks=seq(0, 100, 10), expand=c(0, 0)) +
  scale_y_discrete(expand=expansion(add=c(0.3, 0.3))) +
  labs(title="Between County vs. Within County Variation by Task Group",
       x="Share of Variance", y=NULL) +
  # expand=c(0,0) runs the bars flush to 100 with no trailing gap; the right plot
  # margin holds the 100 tick label, which would otherwise clip at the panel edge
  theme(legend.position="bottom", plot.margin=margin(10, 16, 8, 10))

The Census occupation coding change between 2009 and 2010, described in <a href="#sec-data" class="quarto-xref">Section 3</a>, inflates apparent within county variation for the routine cognitive group. That inflation comes from how the source data were coded rather than from any property of the counties, and the figure is computed on the panel restricted to 2010 onward with each year’s cross county mean removed.

## Model specification

Non-routine cognitive is the reference group. Each remaining coefficient is the change in purchasing power associated with a shift out of non-routine cognitive work and into that group, holding the controls fixed. No coefficient stands alone; each is a comparison against that reference. The group values run on a zero to one scale. Per percentage point figures reported later divide the estimated coefficients by 100.

Year indicators capture conditions common to all counties within a year. These include the 2008 recession, the recovery through the 2010s, and the national price movement described in <a href="#sec-data" class="quarto-xref">Section 3</a>. The estimated year coefficients rise steadily across the panel under both specifications (<a href="#fig-year-effects" class="quarto-xref">Figure 7</a>), and we treat them as nuisance parameters rather than as evidence of rising purchasing power.

In [16]:
# year coefficients from both specifications, converted to their reporting units:
# dollars for the level model, percent for the log model via (exp(b)-1)*100
def _year_frame(m, pct=False):
    co=m.params.filter(like="year_"); ci=m.conf_int().filter(like="year_", axis=0)
    d=pd.DataFrame({"year":[int(s.split("_")[1]) for s in co.index],
                    "estimate":co.values, "lo":ci[0].values, "hi":ci[1].values})
    if pct:
        for c in ["estimate","lo","hi"]: d[c]=(np.exp(d[c])-1)*100
    return d

year_eff_level=_year_frame(model)
year_eff_log=_year_frame(model_log, pct=True)

In [17]:
%%R -i year_eff_level -i year_eff_log -w 8 -h 6.5 -u in -r 150 -b transparent
year_panel <- function(d, ylab, lab_fn) {
  ggplot(d, aes(x=year, y=estimate)) +
    geom_hline(yintercept=0, linetype="dashed", color=GREY, linewidth=0.4) +
    geom_ribbon(aes(ymin=lo, ymax=hi), fill=BLUE, alpha=0.18) +
    geom_line(color=BLUE, linewidth=0.7) +
    geom_point(color=BLUE, size=1.8) +
    scale_y_continuous(labels=lab_fn) +
    # a tick per study year rather than every fourth, scale unchanged
    scale_x_continuous(breaks=scales::pretty_breaks(n=8)) +
    labs(x=NULL, y=ylab)
}

p_lvl <- year_panel(year_eff_level, "Difference in Purchasing Power", dollar_axis) +
  labs(title="Level Specification")
p_log <- year_panel(year_eff_log, "Percent Difference", function(v) sprintf("%g%%", v)) +
  labs(title="Log Specification")

(p_lvl / p_log) +
  plot_annotation(title="Year Effects Relative to 2008",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B"))) &
  theme(panel.grid.major=element_blank(), panel.grid.minor=element_blank())

Standard errors are clustered by county, because repeated observations of the same county across fifteen years are not independent draws. A Durbin Watson statistic of 0.496 is consistent with that dependence. <a href="#sec-results" class="quarto-xref">Section 5</a> reports the estimated coefficients.

### Collinearity and the reference group

The unnormalized employment totals of <a href="#eq-totals" class="quarto-xref">Equation 1</a> cannot support a regression. A variance inflation factor measures how much a coefficient is destabilized by its correlation with the other inputs, and a value above 10 is the conventional threshold for concern. The four totals carry factors between 13 and 27, because all four are employment counts that scale with county size and therefore move together. Fitted on those totals, the coefficients were unstable and their standard errors were large relative to their magnitudes, both standard symptoms of multicollinearity. The specification estimated fixes this twice over, as <a href="#fig-vif" class="quarto-xref">Figure 8</a> shows. Normalizing by the four group total, as <a href="#sec-data" class="quarto-xref">Section 3</a> describes, removes the shared scale, and dropping non-routine cognitive as the reference group removes the constraint that the four proportions sum to one. Every factor on the estimated specification sits between 1.2 and 1.6.

In [18]:
raw=pd.read_sql("""
    select routine_cognitive, routine_manual, non_routine_cognitive, non_routine_manual
    from county_task_exposure
    """, engine)

raw_cols=["routine_cognitive","routine_manual","non_routine_cognitive","non_routine_manual"]
X_raw=sm.add_constant(raw[raw_cols])
vif_raw=pd.Series([variance_inflation_factor(X_raw.values, i) for i in range(1, X_raw.shape[1])],
                  index=raw_cols)

X_vif_level=sm.add_constant(reg[reg_cols])
vif_share=pd.Series([variance_inflation_factor(X_vif_level.values, i) for i in range(1, X_vif_level.shape[1])],
                    index=reg_cols)

group_display=["Routine Cognitive","Routine Manual","Non-Routine Cognitive","Non-Routine Manual"]

vif_dot_df=pd.concat([
    pd.DataFrame({"group":group_display, "vif":vif_raw.values,
                  "spec":"Unnormalized Employment Totals"}),
    pd.DataFrame({"group":["Routine Cognitive","Routine Manual","Non-Routine Manual"],
                  "vif":vif_share[["routine_cognitive_share","routine_manual_share",
                                   "non_routine_manual_share"]].values,
                  "spec":"Group Proportions, as Estimated"}),
], ignore_index=True)

# wide form for the before-to-after arrow on each row (excludes the reference group,
# which has no estimated point)
vif_seg_df=pd.DataFrame({
    "group":["Routine Cognitive","Routine Manual","Non-Routine Manual"],
    "raw":vif_raw[["routine_cognitive","routine_manual","non_routine_manual"]].values,
    "share":vif_share[["routine_cognitive_share","routine_manual_share",
                       "non_routine_manual_share"]].values,
})

In [19]:
%%R -i vif_dot_df -w 8 -h 4 -u in -r 150 -b transparent
vif_dot_df$group <- factor(vif_dot_df$group,
    levels=rev(c("Routine Cognitive","Routine Manual",
                 "Non-Routine Cognitive","Non-Routine Manual")))
vif_dot_df$spec <- factor(vif_dot_df$spec,
    levels=c("Group Proportions, as Estimated","Unnormalized Employment Totals"),
    labels=c("Group proportions","Employment counts"))

ggplot(vif_dot_df, aes(x=vif, y=group)) +
  # bold black thresholds, heavier than the connector segments so they read as
  # reference lines rather than as part of the data
  geom_vline(xintercept=5, linetype="dotted", linewidth=1.1, color="#0B0B0B") +
  geom_vline(xintercept=10, linetype="dashed", linewidth=1.2, color="#0B0B0B") +
  annotate("text", x=5, y=Inf, label="VIF = 5", hjust=-0.15, vjust=1.4, color="#0B0B0B", size=3.1, fontface="bold") +
  annotate("text", x=10, y=Inf, label="VIF = 10", hjust=-0.15, vjust=1.4, color="#0B0B0B", size=3.3, fontface="bold") +
  geom_line(aes(group=group), color="#0B0B0B", linewidth=0.6) +
  geom_point(aes(color=spec), size=3.2) +
  geom_text(aes(label=sprintf("%.1f", vif), color=spec), vjust=-1.1, size=3.2, show.legend=FALSE) +
  scale_x_continuous(limits=c(0, 30), breaks=seq(0, 30, by=2.5)) +
  scale_y_discrete(labels=function(x) ifelse(x=="Non-Routine Cognitive", "Non-Routine Cognitive *", x)) +
  scale_color_manual(values=c("Group proportions"=GREEN,
                              "Employment counts"=ORANGE)) +
  labs(title="Variance Inflation Factors by Task Specification",
       x="Variance Inflation Factor", y=NULL, color=NULL) +
  theme(legend.position="bottom",
        plot.title=element_text(face="bold", hjust=0.5),
        panel.grid.major=element_blank(), panel.grid.minor=element_blank(),
        axis.ticks.x=element_line(color="#0B0B0B", linewidth=0.6),
        axis.ticks.length.x=unit(6, "pt"))

## Specification diagnostics

### Distribution of the analytical variables

With the specification set, we turn to the shape of the variables that enter it. <a href="#fig-dists" class="quarto-xref">Figure 9</a> shows the distributions of purchasing power and the four task groups across the task group panel, looking for the skewness and outliers that would shape the diagnostics below.

In [20]:
dist_label_map={
    "affordability_salary":"Purchasing Power ($)",
    "routine_cognitive_share":"Routine Cognitive",
    "routine_manual_share":"Routine Manual",
    "non_routine_cognitive_share":"Non-Routine Cognitive",
    "non_routine_manual_share":"Non-Routine Manual",
}
dists_df=df.melt(value_vars=list(dist_label_map.keys()),
                 var_name="variable", value_name="value")
dists_df["variable"]=dists_df["variable"].map(dist_label_map)

In [21]:
%%R -i dists_df -w 10 -h 5.5 -u in -r 150 -b transparent
pp_vals <- subset(dists_df, variable=="Purchasing Power ($)")$value
pp_ylim <- c(0, max(pp_vals) * 1.02)

p_pp <- ggplot(subset(dists_df, variable=="Purchasing Power ($)"), aes(x=variable, y=value)) +
  geom_violin(fill=BLUE, color=BLUE, alpha=0.35, linewidth=0.6, trim=TRUE) +
  geom_boxplot(width=0.12, fill="white", color="grey30", alpha=0.7,
               outlier.size=0.8, outlier.alpha=0.5) +
  scale_y_continuous(labels=dollar_axis, breaks=scales::pretty_breaks(n=7)) +
  coord_cartesian(ylim=pp_ylim) +
  labs(title="Purchasing Power", x=NULL, y="Purchasing Power") +
  theme(axis.text.x=element_blank(), axis.ticks.x=element_blank(),
        plot.margin=margin(3, 6, 3, 6)) +
  theme(plot.title=element_text(size=11))

task_box_df <- subset(dists_df, variable %in% c("Routine Cognitive", "Routine Manual",
                                                 "Non-Routine Cognitive", "Non-Routine Manual"))
task_box_df$variable <- factor(task_box_df$variable,
    levels=c("Non-Routine Cognitive", "Non-Routine Manual", "Routine Cognitive", "Routine Manual"))

p_tasks <- ggplot(task_box_df, aes(x=variable, y=value, fill=variable, color=variable)) +
  geom_violin(alpha=0.35, linewidth=0.6, trim=TRUE) +
  geom_boxplot(width=0.12, fill="white", alpha=0.7,
               outlier.size=0.6, outlier.alpha=0.4) +
  scale_fill_manual(values=TASK_COLORS, guide="none") +
  scale_color_manual(values=TASK_COLORS, guide="none") +
  scale_y_continuous(labels=percent, breaks=scales::pretty_breaks(n=7)) +
  coord_cartesian(ylim=c(0.05, 0.65)) +
  scale_x_discrete(labels=function(x) gsub(" ", "\n", x)) +
  labs(title="Task Groups", x=NULL, y=NULL) +
  theme(axis.text.x=element_text(size=8))

(p_pp | p_tasks) + plot_layout(widths=c(1, 1.6)) +
  plot_annotation(title="Distribution of Purchasing Power and Task Groups",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B"))) &
  theme(panel.grid.major=element_blank(), panel.grid.minor=element_blank())

Purchasing power is right skewed, with a skewness of 1.17. The center of the distribution sits near \$57,000, and a long tail of high purchasing power counties extends above it. That skew is why the diagnostics below concentrate on the extremes.

### Residual and quantile diagnostics

A linear model assumes a constant rate of association across the data range and errors that are symmetric and evenly spread around the fit. Three diagnostics test those assumptions. They are the residuals plotted against fitted values, a quantile quantile (QQ) plot of the residuals, and the variance inflation factors already reported. The raw scale model shows clear systematic curvature in the residuals and a substantial upper tail departure in the QQ plot (<a href="#fig-diagnostics" class="quarto-xref">Figure 12</a>, Panels A and B). The model underestimates both the lowest and the highest purchasing power counties while fitting the middle well, and the upper tail departure reflects a residual skew of 1.01.

The level model still supplies the interpretable dollar estimates we report. Its own diagnostics show that the data hold structure a straight line in dollars cannot represent. Curvature in the residual smoother and widening spread with fitted values point to a specific failure in the fixed slope specification, and each model that follows relaxes the assumption behind it.

### Influential points

A fourth diagnostic, Cook’s distance, checks whether any single county year is disproportionately steering the level specification’s coefficients. No observation crosses the conventional threshold of 1.0 for a point that should always be investigated; the largest value is 0.009. <a href="#fig-cooks-distance" class="quarto-xref">Figure 10</a> ranks the 10 counties with the largest Cook’s distance in any single year, taking each county’s own maximum across its years in the panel. The figure also plots every year’s value behind that maximum. Counties reach the ranking by two different routes. Loudoun County, Virginia, the largest by far, gets there through real year to year volatility, while several counties lower down, such as McKinley County, New Mexico, sit consistently high across nearly every year. Loudoun and Stafford County, Virginia are both high purchasing power counties in the Washington D.C. exurbs; Williamson County, Tennessee is a high purchasing power Nashville exurb; and Apache County, Arizona is a low purchasing power county on the Navajo Nation. These are the same high end outliers <a href="#fig-dists" class="quarto-xref">Figure 9</a> already shows in the purchasing power distribution.

In [22]:
cooks_infl=model.get_influence()
reg["cooks_d"]=cooks_infl.cooks_distance[0]

county_state_names=pd.read_sql(
    "select c.county_fips, c.county_name, s.state_name "
    "from county c join state s on c.state_code=s.state_code", engine)

# each county's own largest Cook's distance across its years in the panel,
# so a persistently influential county appears once rather than once per year
county_max_cooks=reg.groupby("county_fips")["cooks_d"].max().reset_index()
top10_fips=county_max_cooks.nlargest(10, "cooks_d")["county_fips"]

# every year's value for those 10 counties, not just each county's max,
# so the figure can show the full range rather than a single collapsed number
top_influential=(reg[reg["county_fips"].isin(top10_fips)][["county_fips","cooks_d"]]
                  .merge(county_state_names, on="county_fips", how="left"))
top_influential["label"]=top_influential["county_name"]+", "+top_influential["state_name"]
influence_plot_df=top_influential[["label","cooks_d"]]
influence_range_df=(influence_plot_df.groupby("label")["cooks_d"]
                     .agg(min_d="min", max_d="max").reset_index()
                     .sort_values("max_d"))
influence_plot_df["label"]=pd.Categorical(influence_plot_df["label"],
                                          categories=influence_range_df["label"], ordered=True)

# sensitivity check: refit excluding the top 1 percent most influential rows
influence_cutoff=reg["cooks_d"].quantile(0.99)
influence_keep=reg["cooks_d"]<=influence_cutoff
n_excluded=int((~influence_keep).sum())

model_robust=sm.OLS(reg.loc[influence_keep, "affordability_salary"], X_panel.loc[influence_keep]).fit(
    cov_type="cluster", cov_kwds={"groups": reg.loc[influence_keep, "county_fips"]})

sens_labels={
    "routine_cognitive_share":"Routine Cognitive",
    "routine_manual_share":"Routine Manual",
    "non_routine_manual_share":"Non-Routine Manual",
    "poverty_rate":"Poverty Rate",
    "unemployment_rate":"Unemployment Rate",
    "log_population":"Log Population",
}
sensitivity_df=pd.DataFrame({
    "Variable": [sens_labels[c] for c in reg_cols],
    "Full Sample": [model.params[c] for c in reg_cols],
    "Excluding Top 1%": [model_robust.params[c] for c in reg_cols],
})
sensitivity_df["Change (%)"]=(sensitivity_df["Excluding Top 1%"]/sensitivity_df["Full Sample"]-1)*100

In [23]:
%%R -i influence_plot_df -i influence_range_df -w 9 -h 5.5 -u in -r 150 -b transparent
influence_plot_df$label <- factor(influence_plot_df$label, levels=influence_range_df$label)
influence_range_df$label <- factor(influence_range_df$label, levels=influence_range_df$label)

ggplot() +
  geom_segment(data=influence_range_df, aes(x=min_d, xend=max_d, y=label, yend=label),
               linewidth=1.2, color="#C8C8C5", lineend="round") +
  geom_point(data=influence_plot_df, aes(x=cooks_d, y=label), size=2.0, color=BLUE, alpha=0.45) +
  geom_point(data=influence_range_df, aes(x=max_d, y=label), size=3.2, color=BLUE) +
  geom_text(data=influence_range_df, aes(x=max_d, y=label, label=sprintf("%.4f", max_d)),
            hjust=-0.35, size=3.2, color="#252525") +
  scale_x_continuous(limits=c(0, 0.011), expand=c(0,0)) +
  labs(title="Ten Highest Cook's Distance Values by County, Level Specification",
       x="Cook's Distance", y=NULL)

No single point threatens the fit, so the more informative check is a refit. We drop the most influential 1 percent of the panel, 120 county year observations, and estimate the model again. <a href="#tbl-influence-sensitivity" class="quarto-xref">Table 2</a> reports the result. The task group coefficients move by single digit to low double digit percentages, and the poverty coefficient barely moves at all. The associations this study leans on hardest are not artifacts of a handful of extreme counties. Unemployment rate is the exception. It is already the smallest and least precisely estimated of the controls, as <a href="#sec-results" class="quarto-xref">Section 5</a> reports, and its coefficient shifts by roughly a third. That sensitivity fits the weak evidence unemployment rate carries throughout this study, and does not indicate a problem specific to this check. A second stability check runs the same logic over time. We refit the main estimates on pre pandemic (2010 to 2019) and post pandemic (2021 to 2023) windows, and <a href="#sec-results" class="quarto-xref">Section 5</a> reports coefficient stability across them.

In [24]:
style_table(GT(sensitivity_df)
  .fmt_currency(columns=["Full Sample","Excluding Top 1%"], decimals=0)
  .fmt_number(columns="Change (%)", decimals=1)
  .tab_source_note(f"{n_excluded} of {len(reg)} county year observations excluded, the top 1% by Cook's distance."))

## Log respecification

The level model’s failure suggests that a constant dollar association is a poor description of the relationship. Logging the outcome tests the alternative, which is that the associations are proportional. Under that description the same shift in an input moves a high purchasing power county by more dollars than a low one. If it fits better, the transformation should resolve the residual pattern the level model leaves behind. <a href="#fig-log-transform" class="quarto-xref">Figure 11</a> shows the outcome before (Panel A) and after (Panel B) the transformation.

In [25]:
level_skew=float(reg["affordability_salary"].skew())
log_skew=float(reg["actual_log"].skew())
logt_df=pd.concat([
    pd.DataFrame({"value":reg["affordability_salary"], "dist":"level"}),
    pd.DataFrame({"value":reg["actual_log"], "dist":"log"}),
], ignore_index=True)

In [26]:
%%R -i logt_df -i level_skew -i log_skew -w 11 -h 4.8 -u in -r 150 -b transparent

lv <- subset(logt_df, dist=="level")$value
lg <- subset(logt_df, dist=="log")$value

# density curve with the median and mean marked; the gap between the two lines is
# the skew each panel reports, made visible rather than only stated
dens_panel <- function(v, title, skew, xlab, dollars, fill) {
  # density on the dollar scale is ~2e-05, which R prints in scientific notation;
  # force plain decimals there and leave the log panel on default labels
  y_labels <- if (dollars) function(v) sprintf("%.5f", v) else waiver()
  d <- density(v)
  dd <- data.frame(x=d$x, y=d$y)
  md <- median(v); mn <- mean(v); ytop <- max(dd$y)
  p <- ggplot(dd, aes(x=x, y=y)) +
    geom_area(fill=fill, alpha=0.9) +
    geom_line(color="#0B0B0B", linewidth=0.7) +
    geom_vline(xintercept=md, color="#0B0B0B", linewidth=0.6) +
    geom_vline(xintercept=mn, color="#0B0B0B", linewidth=0.6, linetype="dashed") +
    annotate("text", x=md, y=ytop*1.06, label="Median", size=2.9, hjust=1.12, color="#0B0B0B") +
    annotate("text", x=mn, y=ytop*1.06, label="Mean", size=2.9, hjust=-0.12, color="#0B0B0B") +
    scale_y_continuous(labels=y_labels, expand=expansion(mult=c(0, 0.12))) +
    labs(title=title, subtitle=sprintf("Skewness = %.2f", skew), x=xlab, y="Density") +
    theme(plot.title=element_text(size=11),
          plot.subtitle=element_text(size=9, hjust=0.5, face="italic"))
  if (dollars) p <- p + scale_x_continuous(labels=dollar_axis)
  p
}

p_level <- dens_panel(lv, "Original Scale", level_skew, "Purchasing Power", TRUE, ORANGE)
p_log   <- dens_panel(lg, "Log Scale", log_skew, "Log Purchasing Power", FALSE, GREEN)

(p_level | p_log) +
  plot_annotation(title="Distribution of Purchasing Power Before and After Log Transformation",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B"))) &
  theme(panel.grid.major=element_blank(), panel.grid.minor=element_blank())

Refitting the same specification on log purchasing power resolves most of the failure. <a href="#fig-diagnostics" class="quarto-xref">Figure 12</a> compares the residual and QQ diagnostics for both specifications directly. After logging purchasing power (Panels C and D), the residual pattern becomes substantially flatter and the QQ points track the reference line more closely. Some tail deviation remains, but the residual skew falls from 1.01 to 0.09. The log linear fit is the baseline against which we evaluate the flexible models below.

In [27]:
resid_df=reg[["fitted","resid"]].copy()
resid_log_df=reg[["fitted_log","resid_log"]].copy()

In [28]:
%%R -i resid_df -i resid_log_df -w 9 -h 7 -u in -r 150 -b transparent
resid_df$std_resid <- as.numeric(scale(resid_df$resid))
resid_log_df$std_resid <- as.numeric(scale(resid_log_df$resid_log))

p1 <- ggplot(resid_df, aes(x=fitted, y=resid)) +
  geom_bin2d(bins=55, aes(fill=after_stat(count))) +
  scale_fill_gradient(low="#E8EEF4", high=BLUE, guide="none") +
  geom_hline(yintercept=0, linewidth=0.4, linetype="dashed", color="grey40") +
  geom_smooth(method="loess", formula=y~x, se=FALSE, color=RED, linewidth=0.9) +
  scale_x_continuous(labels=dollar_axis) +
  scale_y_continuous(labels=dollar_axis, breaks=scales::pretty_breaks(n=7)) +
  labs(x="Fitted Purchasing Power", y="Residual, Raw", title="Residuals vs. Fitted") +
  theme(plot.title=element_text(size=11))

p2 <- ggplot(resid_df, aes(sample=std_resid)) +
  stat_qq(color=BLUE, alpha=0.35, size=0.8) +
  stat_qq_line(color="#0B0B0B", linewidth=0.8) +
  scale_y_continuous(breaks=scales::pretty_breaks(n=7)) +
  labs(x="Theoretical Quantiles", y="Standardized Residual", title="Normal Q-Q") +
  theme(plot.title=element_text(size=11))

p3 <- ggplot(resid_log_df, aes(x=fitted_log, y=resid_log)) +
  geom_bin2d(bins=55, aes(fill=after_stat(count))) +
  scale_fill_gradient(low="#E8F4EE", high=GREEN, guide="none") +
  geom_hline(yintercept=0, linewidth=0.4, linetype="dashed", color="grey40") +
  geom_smooth(method="loess", formula=y~x, se=FALSE, color=RED, linewidth=0.9) +
  scale_y_continuous(breaks=scales::pretty_breaks(n=7)) +
  labs(x="Fitted Log Purchasing Power", y="Residual, Log")

p4 <- ggplot(resid_log_df, aes(sample=std_resid)) +
  stat_qq(color=GREEN, alpha=0.35, size=0.8) +
  stat_qq_line(color="#0B0B0B", linewidth=0.8) +
  scale_y_continuous(breaks=scales::pretty_breaks(n=7)) +
  labs(x="Theoretical Quantiles", y="Standardized Residual")

# horizontal gridlines off on every panel; the dashed zero line and the QQ
# reference line are the references that matter here
grid_off <- theme(panel.grid.major.y=element_blank(), panel.grid.minor.y=element_blank())
p1 <- p1 + grid_off; p2 <- p2 + grid_off; p3 <- p3 + grid_off; p4 <- p4 + grid_off

(p1 | p2) / (p3 | p4) +
  plot_annotation(title="Regression Diagnostics, Level vs. Log Specification",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B")))

<a href="#fig-binned-scale" class="quarto-xref">Figure 13</a> puts the same comparison in the units of the outcome. On the level scale the model tracks purchasing power closely through the middle of the distribution, where most counties sit, while both tail bins come in roughly \$8,000 to \$9,000 above their predictions. On the log scale the points track the line throughout.

In [29]:
reg.loc[:, "bin"]=pd.qcut(reg["fitted"], 20, labels=False)
binned=reg.groupby("bin")[["fitted","affordability_salary"]].mean()
reg.loc[:, "bin_log"]=pd.qcut(reg["fitted_log"], 20, labels=False)
binned_log=reg.groupby("bin_log")[["fitted_log","actual_log"]].mean()

pts_df=pd.concat([
    pd.DataFrame({"fitted":reg["fitted"], "actual":reg["affordability_salary"],
                  "target":"Level Target"}),
    pd.DataFrame({"fitted":reg["fitted_log"], "actual":reg["actual_log"],
                  "target":"Log Target"}),
], ignore_index=True)
bins_df=pd.concat([
    pd.DataFrame({"fitted":binned["fitted"], "actual":binned["affordability_salary"],
                  "target":"Level Target"}),
    pd.DataFrame({"fitted":binned_log["fitted_log"], "actual":binned_log["actual_log"],
                  "target":"Log Target"}),
], ignore_index=True)

In [30]:
%%R -i pts_df -i bins_df -w 10 -h 5 -u in -r 150 -b transparent
dollar_axis <- function(v) ifelse(is.na(v), "", ifelse(v == 0, "$0",
  sprintf("%s$%s", ifelse(v < 0, "-", ""), formatC(abs(v), format="d", big.mark=","))))

# patchwork rather than facets: the two panels carry different units, so each needs
# its own axis titles, and only separate plots can take A) and B) tags
panel <- function(tg, col, xlab, ylab, title, dollars) {
  p <- ggplot(subset(pts_df, target==tg), aes(x=fitted, y=actual)) +
    geom_point(alpha=0.05, size=0.3, color="grey50") +
    geom_abline(slope=1, intercept=0, linewidth=0.9, color="#0B0B0B") +
    geom_point(data=subset(bins_df, target==tg), size=2.4, color=col) +
    scale_x_continuous(breaks=scales::pretty_breaks(n=5),
                       labels=if (dollars) dollar_axis else waiver()) +
    scale_y_continuous(breaks=scales::pretty_breaks(n=5),
                       labels=if (dollars) dollar_axis else waiver()) +
    labs(x=xlab, y=ylab, title=title) +
    theme(plot.title=element_text(size=11))
  p
}

p_level <- panel("Level Target", ORANGE, "Fitted Purchasing Power",
                 "Actual Purchasing Power", "Level Target", TRUE)
p_log   <- panel("Log Target", GREEN, "Fitted Log Purchasing Power",
                 "Actual Log Purchasing Power", "Log Target", FALSE)

(p_level | p_log) +
  plot_annotation(title="Binned Fit Accuracy, Level vs. Log Specification",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B"))) &
  theme(panel.grid.major=element_blank(), panel.grid.minor=element_blank())

The smoothed residual line does not lie perfectly flat even after the transformation. What remains could be nonlinearity, interactions among the inputs, or a variable the model does not contain, and the residuals alone cannot separate the three. That is exactly the situation a flexible model is built for. A random forest searches all three at once without being told in advance which to look for, and it tests the possibilities the diagnostics raise but cannot settle. The log linear fit also sets the bar the forest has to clear, which is finding signal in held out counties that the fit does not already capture.

## Predictive modeling

Model complexity follows a ladder, from linear models through tree ensembles to neural networks. The rule is to start with the simplest model that could plausibly work, move up a rung only while held out error improves, and stop when training and validation performance agree. Failed diagnostics on the level model justified the first step, the log respecification. Structured residuals that remain justify testing the next rung, a random forest, which represents interactions and curvature without requiring either to be specified in advance. A small neural network sits one rung above the forest and bounds the search. If the forest finds no additional signal, the network confirms whether that ceiling is real. Every rung is scored on the same held out counties from the grouped split, which makes a gain at any rung attributable to the model rather than to the rows it saw. Under this rule, finding no improvement is itself a result, because it establishes that the association is close to proportional and that the log linear model is the right stopping point.

We reuse the county year panel assembled above, extended with state identifiers for the comparison below.

In [31]:
# state_code isn't in the panel regression's df, add it here
state_lookup=pd.read_sql(
    "select c.county_fips, c.state_code, s.state_name "
    "from county c join state s on c.state_code=s.state_code", engine
)
state_name_map=state_lookup.drop_duplicates("state_code").set_index("state_code")["state_name"]
df_ml=df.merge(state_lookup, on="county_fips", how="left")

### Evaluation design

#### Feature set and reference groups

In [32]:
feature_cols=[
    "routine_cognitive_share",
    "routine_manual_share",
    "non_routine_cognitive_share",
    "non_routine_manual_share",
    "poverty_rate",
    "unemployment_rate",
    "year",
    "state_code",
]

model_df=df_ml.dropna(subset=feature_cols).reset_index(drop=True).copy()

We define each task group as its proportion of employment across the four categories, and the four values sum to one. As with the panel regression, that constraint makes non-routine cognitive the omitted reference, and the coefficients read the same way.

#### Grouped train and test split

This is not a forecasting exercise, and a time ordered split is not required for its usual reason. A grouped split is still necessary, because a county’s economic profile barely moves year to year, which makes its 2018 and 2019 rows near duplicates. If two adjacent rows land on opposite sides of the split, the model can recall a county rather than learn a relationship. We group by county and split on row membership before any further feature engineering. The split assigns 9,565 county year observations to training and holds out the remaining 2,418, with no county contributing rows to both. Every test set prediction therefore comes from a county the model has seen nothing of, which tests whether the pattern generalizes to new places.

In [33]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

target_col="affordability_salary"
groups=model_df["county_fips"]

gss=GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx=next(gss.split(model_df, groups=groups))

groups_train=groups.iloc[train_idx]

assert len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))==0

#### Year and state indicators

Treating year as a factor rather than as a numeric variable gives each year its own coefficient, with no assumption about ordering or spacing between them. The earliest year in the panel is held out as the reference, and every other year reads against it. States enter the same way, with one state held out and named below. Every state with fewer than five distinct counties in the panel is relabeled as OTHER, since those states cannot support a stable coefficient of their own.

In [34]:
model_df["year"]=model_df["year"].astype("category")
model_df["state_code"]=model_df["state_code"].astype("category")

REFERENCE_YEAR=model_df["year"].cat.categories.min()  # earliest year as reference

# pool states with too few counties to support a stable, independent coefficient
county_counts=model_df.groupby("state_code", observed=True)["county_fips"].nunique()
THIN_STATE_THRESHOLD=5
thin_states=county_counts[county_counts<THIN_STATE_THRESHOLD].index.tolist()

model_df["state_code_grouped"]=model_df["state_code"].astype(str)
model_df["state_code_grouped"]=model_df["state_code_grouped"].where(
    ~model_df["state_code_grouped"].isin(thin_states), "OTHER"
)
model_df["state_code_grouped"]=model_df["state_code_grouped"].astype("category")

state_affordability=model_df.groupby("state_code_grouped", observed=True)["affordability_salary"].mean().sort_values()

median_state=state_affordability.index[len(state_affordability)//2]
REFERENCE_STATE=median_state
# resolved once here so the prose, the figure accent, and the held out indicator
# all name the same state rather than each re-deriving it
reference_state_label=state_name_map.get(REFERENCE_STATE, str(REFERENCE_STATE))

year_dummies=pd.get_dummies(model_df["year"], prefix="year")
year_dummies=year_dummies.drop(columns=[f"year_{REFERENCE_YEAR}"])

state_dummies=pd.get_dummies(model_df["state_code_grouped"], prefix="state")
state_dummies=state_dummies.drop(columns=[f"state_{REFERENCE_STATE}"])

model_df=pd.concat([model_df, year_dummies, state_dummies], axis=1)

year_cols_model=list(year_dummies.columns)
state_cols_model=list(state_dummies.columns)

The reference state is the one whose mean purchasing power falls at the median of the state means, which is Texas (<a href="#fig-state-afford" class="quarto-xref">Figure 14</a>). It is not a cost of living outlier in either direction, and it holds considerable within state diversity. Major metropolitan areas sit alongside a large number of rural counties. Reading every other state against that baseline keeps the coefficients interpretable. Most state coefficients sit within a few percent of the reference, though the largest departures reach 17 percent. The block contributes little to held out accuracy once the economic controls and time are present, as the permutation results below show.

In [35]:
state_df=state_affordability.reset_index()
state_df.columns=["state","mean_pp"]
state_df["state_label"]=state_df["state"].map(state_name_map)
# the pooled thin-state bucket is not a state; excluded from the ranking display below
state_df=state_df[state_df["state"]!="OTHER"].copy()
state_median=float(state_affordability.median())
state_df["comparison"]=state_df["mean_pp"].ge(state_median).map({True:"Above Average", False:"Below Average"})

In [36]:
%%R -i state_df -i state_median -i reference_state_label -w 8 -h 10 -u in -r 150 -b transparent
state_df$state_label <- factor(state_df$state_label, levels=state_df$state_label)
# accent the state the models hold out as the reference. Matching on the label rather
# than re-deriving the median keeps the figure and the fitted models in agreement;
# %in% leaves every point unaccented if the reference is the pooled bucket
ACCENT <- BLUE
ref_i <- which(state_df$state_label == reference_state_label)
state_df$pt_color <- ifelse(seq_len(nrow(state_df)) %in% ref_i, ACCENT, "#0B0B0B")
state_df$pt_size  <- ifelse(seq_len(nrow(state_df)) %in% ref_i, 3.8, 2.6)
median_label <- sprintf("Median: $%.0fk", state_median/1000)
n_states <- length(levels(state_df$state_label))
pp_range <- range(state_df$mean_pp)

# axis starts at $30k rather than zero. The Lie Factor concern that motivated a
# zero anchor does not bind here: each segment runs from the median line to the
# state's value, so its length encodes deviation from the median rather than
# absolute magnitude, and moving the left edge rescales every segment equally.
x_lo <- 30000
x_hi <- pp_range[2] * 1.05

# round $20k breaks. Offsets from the median were tried and reverted: anchoring
# every tick to the median produced labels like $4.47k and $9.47k that collided.
# The median keeps its own dotted line and its labeled annotation instead.
state_breaks <- seq(0, 200000, by=10000)
state_breaks <- state_breaks[state_breaks >= x_lo & state_breaks <= x_hi]

# annotation centered within each visual half of the zero-anchored panel
low_mid  <- (x_lo + state_median) / 2
high_mid <- (state_median + x_hi) / 2

ggplot(state_df, aes(x=mean_pp, y=state_label)) +
  annotate("rect", xmin=-Inf, xmax=state_median, ymin=-Inf, ymax=Inf,
           fill="#C0392B", alpha=0.10) +
  annotate("rect", xmin=state_median, xmax=Inf, ymin=-Inf, ymax=Inf,
           fill=GREEN, alpha=0.10) +
  # wrapped onto two lines so neither label crosses the median line or runs off
  # the panel edge, which the single line versions did at size 6
  annotate("text", x=low_mid, y=n_states / 2, label="Lower\nPurchasing\nPower",
           color="#C0392B", fontface="bold", size=5.5, hjust=0.5, vjust=0.5,
           alpha=1, lineheight=0.95) +
  annotate("text", x=high_mid, y=n_states / 2, label="Higher\nPurchasing\nPower",
           color=GREEN, fontface="bold", size=5.5, hjust=0.5, vjust=0.5,
           alpha=1, lineheight=0.95) +
  geom_vline(xintercept=state_median, linetype="dotted", linewidth=0.8, color=ACCENT) +
  annotate("text", x=state_median + (x_hi - x_lo) * 0.012, y=n_states + 1.9, label=median_label,
           size=3, color="black", hjust=0, vjust=0) +
  geom_segment(aes(x=state_median, xend=mean_pp, yend=state_label), linewidth=0.9, alpha=0.5, color="black") +
  geom_point(aes(color=pt_color, size=pt_size)) +
  scale_color_identity() +
  scale_size_identity() +
  # coord_cartesian rather than limits=, which would drop any state below the floor
  scale_x_continuous(labels=label_dollar(scale=1e-3, suffix="k"), breaks=state_breaks) +
  coord_cartesian(xlim=c(x_lo, x_hi)) +
  scale_y_discrete(expand=expansion(add=c(0.6, 2.6))) +
  labs(title="Mean Purchasing Power by State",
       x="Mean Purchasing Power", y=NULL) +
  # no gridlines at all, so nothing crosses the two watermark labels
  theme(axis.text.y=element_text(size=7), panel.grid.major=element_blank(),
        panel.grid.minor=element_blank(),
        plot.margin=margin(10,10,10,10))

#### Collinearity of the indicator blocks

The reference group logic of <a href="#fig-vif" class="quarto-xref">Figure 8</a> applies here unchanged. What is new in this feature set is the indicator blocks, and <a href="#tbl-vif-ml" class="quarto-xref">Table 3</a> shows they introduce no collinearity of their own.

In [37]:
exposure_check_cols=["routine_cognitive_share",
    "routine_manual_share",
    "non_routine_cognitive_share",
    "non_routine_manual_share"]

REFERENCE_EXPOSURE="non_routine_cognitive_share"
exposure_cols_model=[c for c in exposure_check_cols if c!=REFERENCE_EXPOSURE]
core_features=exposure_cols_model+["poverty_rate", "unemployment_rate"]

In [38]:
vif_features=core_features+year_cols_model+state_cols_model

X_vif=sm.add_constant(model_df[vif_features].astype(float))

vif_data=pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
})
vif_top=vif_data[vif_data["feature"].isin(year_cols_model+state_cols_model)].sort_values("VIF", ascending=False).head(10).copy()
vif_top["Indicator"]=(vif_top["feature"]
                      .str.replace("year_", "", regex=False)
                      .str.replace("state_", "State ", regex=False))
vif_tbl_df=vif_top[["Indicator","VIF"]].sort_values("VIF", ascending=False)

In [39]:
style_table(GT(vif_tbl_df.reset_index(drop=True))
  .fmt_number(columns="VIF", decimals=2)
  .cols_align(align="center", columns=["Indicator","VIF"])
  .tab_source_note("Computed on the full feature matrix after dropping the reference task group."))

In [40]:
# feature matrix, built now that the year and state indicators and the reference
# group logic above are settled; sliced using the grouped split made earlier
feature_cols_reg=core_features+year_cols_model+state_cols_model

X_ml=model_df[feature_cols_reg].astype(float)
y=model_df[target_col]

X_train, X_test=X_ml.iloc[train_idx], X_ml.iloc[test_idx]
y_train, y_test=y.iloc[train_idx], y.iloc[test_idx]

# baseline feature set (no year/state), reusing the SAME row split as the full model
feature_cols_base=exposure_cols_model+["poverty_rate", "unemployment_rate"]
X_base=model_df[feature_cols_base].astype(float)
X_train_base, X_test_base=X_base.iloc[train_idx], X_base.iloc[test_idx]

### Linear benchmark

This specification is a predictive benchmark rather than a second inferential model. It reuses the diagnostic lessons the panel regression above already established, fit now on the training split alone. That shared split makes it directly comparable to the random forest and neural network that follow. We report two versions of each linear model, estimated by ordinary least squares (OLS). The baseline uses the task groups with poverty and unemployment only, while the full model adds year and state controls. Reporting both makes visible exactly what changes when time and geography enter. Of the two, the full model is the one carried forward.

#### Baseline specification

In [41]:
X_train_base_sm=sm.add_constant(X_train_base)
ols_model_base=sm.OLS(y_train, X_train_base_sm).fit()

The baseline reaches an R² of 0.766. Every coefficient is negative, which follows from non-routine cognitive serving as the reference. Shifting a county toward any of the other three groups is associated with lower purchasing power, and non-routine manual is the steepest. The coefficients appear alongside the full specifications in <a href="#tbl-ols-comparison" class="quarto-xref">Table 4</a>.

#### Full specification

In [42]:
X_train_sm=sm.add_constant(X_train)
ols_model=sm.OLS(y_train, X_train_sm).fit()

Adding time and geography raises R² from 0.766 to 0.871, which shows that these variables explain variance the task groups do not. The unemployment rate coefficient flips sign, which suggests that the baseline coefficient absorbed some of the geographic and temporal confounding. That is what we would expect, since places and periods with heavy unemployment also tend to be places and periods of low purchasing power. Once time and geography are controlled, the full model’s coefficient reflects the within state year relationship instead. Across the change, the task group coefficients keep their direction and their relative ordering.

The raw target residuals repeat the curvature already documented in <a href="#fig-diagnostics" class="quarto-xref">Figure 12</a> (Panels A and B). The untransformed target defeats a linear model on the training split exactly as it did on the full panel.

#### Full specification on the logged outcome

The binned means in <a href="#fig-binned-scale" class="quarto-xref">Figure 13</a> showed an approximately exponential pattern, and we test whether logging the target makes a linear model appropriate here as well.

In [43]:
assert (y<=0).sum()==0  # confirm log is safe

y_train_log=np.log(y_train)
y_test_log=np.log(y_test)

ols_model_log=sm.OLS(y_train_log, X_train_sm).fit()

X_test_sm=sm.add_constant(X_test, has_constant="add")
y_pred_ols_log=ols_model_log.predict(X_test_sm)
y_pred_ols_log_dollars=np.exp(y_pred_ols_log)

ols_log_test_r2_log=r2_score(y_test_log, y_pred_ols_log)
ols_log_test_r2=r2_score(y_test, y_pred_ols_log_dollars)
ols_log_test_mae=mean_absolute_error(y_test, y_pred_ols_log_dollars)

Logging the target resolves the pattern here the same way it did for the panel regression (<a href="#fig-diagnostics" class="quarto-xref">Figure 12</a>, Panels C and D). A linear model is appropriate on the logged target. That model reaches an in sample R² of 0.913 and a held out R², back transformed into dollars, of 0.896.

#### Coefficient comparison

<a href="#tbl-ols-comparison" class="quarto-xref">Table 4</a> reports the coefficients across the three specifications, and <a href="#sec-results" class="quarto-xref">Section 5</a> shows the same comparison graphically.

In [44]:
log_coefs=ols_model_log.params[feature_cols_base]

# task groups are 0 to 1 scale (1pp=0.01 units); poverty and unemployment are already in point units
unit_per_point={
    "routine_cognitive_share": 0.01,
    "routine_manual_share": 0.01,
    "non_routine_manual_share": 0.01,
    "poverty_rate": 1.0,
    "unemployment_rate": 1.0,
}

pct_change_per_point={
    feat: (np.exp(log_coefs[feat]*unit_per_point[feat])-1)*100
    for feat in feature_cols_base
}

label_map={
    "routine_cognitive_share":"Routine Cognitive",
    "routine_manual_share":"Routine Manual",
    "non_routine_manual_share":"Non-Routine Manual",
    "poverty_rate":"Poverty Rate",
    "unemployment_rate":"Unemployment Rate",
}

task_feats={"routine_cognitive_share","routine_manual_share","non_routine_manual_share"}

comparison_ols=pd.DataFrame({
    "Group": ["Task Groups" if f in task_feats else "Economic Controls"
              for f in feature_cols_base],
    "Variable": [label_map[f] for f in feature_cols_base],
    "Baseline": ols_model_base.params[feature_cols_base].values,
    "Full": ols_model.params[feature_cols_base].values,
    "Coefficient": ols_model_log.params[feature_cols_base].values,
    "Percent per point": [pct_change_per_point[f] for f in feature_cols_base],
})
task_rows=[i for i,g in enumerate(comparison_ols["Group"]) if g=="Task Groups"]

style_table(GT(comparison_ols, rowname_col="Variable", groupname_col="Group")
  .fmt_currency(columns=["Baseline","Full"], decimals=0)
  .fmt_number(columns="Coefficient", decimals=3)
  .fmt_number(columns="Percent per point", decimals=2)
  .cols_label(**{"Percent per point": "% per 1 pp"})
  .tab_spanner(label="Purchasing Power", columns=["Baseline","Full"])
  .tab_spanner(label="Log Transformed Purchasing Power", columns=["Coefficient","Percent per point"])
  .tab_style(style=style.text(weight="bold"),
             locations=loc.body(columns="Percent per point", rows=task_rows))
  .tab_style(style=style.text(align="center", weight="bold"), locations=loc.row_groups())
  .tab_source_note("Non-routine cognitive is the reference task group. Task group coefficients in the three model columns are per unit of group proportion; the final column converts the log coefficients to the percent change in purchasing power associated with a one percentage point shift."))

The sign flip in unemployment rate holds across all three fitted forms. Task group coefficients shrink in magnitude from baseline to raw controls, while the poverty coefficient grows from roughly −1,376 to −1,729. The controls therefore sharpen the task group association rather than displacing it.

Each additional percentage point of poverty rate is associated with roughly 2.9 percent lower purchasing power, holding time, geography, and task groups fixed. That is a considerably larger proportional difference than any single task group carries.

In [45]:
spec_summary = pd.DataFrame({
    "Metric": ["Outcome", "Year and State Controls", "R²",
               "Residual Diagnostics", "Primary Specification"],
    "Baseline": ["Purchasing Power", "No", f"{ols_model_base.rsquared:.3f}", "—", "No"],
    "Full Model": ["Purchasing Power", "Yes", f"{ols_model.rsquared:.3f}", "Failed", "No"],
    "Log Model": ["log(Purchasing Power)", "Yes", f"{ols_model_log.rsquared:.3f}", "Passed", "Yes"],
})

In [46]:
style_table(GT(spec_summary, rowname_col="Metric")
  .tab_style(style=style.text(weight="bold"),
             locations=loc.body(columns="Log Model", rows=[2, 3, 4]))
  .tab_source_note("R² is in sample, fit on the 9,565 training rows only."))

<a href="#tbl-specmatrix" class="quarto-xref">Table 5</a> compares the three specifications. The log model is the one used for inference from here, since the raw target model’s coefficient significance cannot be trusted given its failed diagnostics.

### Random forest

For the random forest we ran a grid search scored by grouped five fold cross validation on the training counties. The grid covered the number of trees (600 and 1,000), maximum depth (10, 20, and unlimited), minimum leaf size (1, 2, and 5), and the feature subsampling rate (all features, half, and the square root). More trees reduce variance, while depth and leaf size supply regularization. The search selected 1,000 trees, unlimited depth, a minimum leaf of 1, and half the features per split.

In [47]:
# grid search actually run once; kept for documentation of the search space,
# not re-executed on render. Best params hardcoded in the following chunk.

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, GroupKFold

rf=RandomForestRegressor(random_state=42, n_jobs=-1)

param_grid={
    "n_estimators": [600, 1000],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2, 5],
    "max_features": [1.0, "sqrt", 0.5],
}

group_cv=GroupKFold(n_splits=5)
gcv=GridSearchCV(rf, param_grid, cv=group_cv, scoring="r2", n_jobs=-1)
gcv.fit(X_train, y_train, groups=groups_train)

In [48]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

# grid search above was run once over the full param_grid; best params found were:
# {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 1000}
best_params={
    "max_depth": None,
    "max_features": 0.5,
    "min_samples_leaf": 1,
    "n_estimators": 1000,
}

best_rf=RandomForestRegressor(**best_params, random_state=42, n_jobs=2)
best_rf.fit(X_train, y_train)

y_pred_rf=best_rf.predict(X_test)
rf_test_r2=r2_score(y_test, y_pred_rf)
rf_test_mae=mean_absolute_error(y_test, y_pred_rf)
rf_train_r2=best_rf.score(X_train, y_train)

The forest reaches a held out R² of 0.876 on the raw target. It does not improve on the log target OLS model, which reaches 0.896 on the same held out counties.

The forest fits the training data far more closely than it fits unseen counties, 0.989 against 0.876. Its additional flexibility captures patterns in the training sample that do not carry over to counties it has not seen.

In [49]:
rf_resid_df=pd.DataFrame({"pred":y_pred_rf, "resid":y_test-y_pred_rf})

In [50]:
%%R -i rf_resid_df -w 7 -h 5 -u in -r 150 -b transparent
ggplot(rf_resid_df, aes(x=pred, y=resid)) +
  geom_point(alpha=0.3, size=0.6, color=PINK) +
  geom_hline(yintercept=0, linetype="dashed", linewidth=0.9, color="#0B0B0B") +
  scale_x_continuous(labels=dollar_axis) +
  scale_y_continuous(labels=dollar_axis) +
  labs(title="Random Forest Residuals vs. Predicted Values",
       x="Predicted Values", y="Residuals") +
  theme(panel.grid.major.y=element_blank())

The curvature that defeated the raw target linear model does not appear here (<a href="#fig-rf-resid" class="quarto-xref">Figure 15</a>). The forest represents that structure internally, without the outcome needing to be transformed first.

In [51]:
perm_imp=permutation_importance(
    best_rf, X_test, y_test, n_repeats=20, random_state=42, n_jobs=1
)

#### Ablation and permutation importance

In [52]:
rf_log=RandomForestRegressor(**best_params, random_state=42, n_jobs=2)
rf_log.fit(X_train, y_train_log)

y_pred_rf_log=rf_log.predict(X_test)
y_pred_rf_log_dollars=np.exp(y_pred_rf_log)

rf_log_test_r2_log=r2_score(y_test_log, y_pred_rf_log)
rf_log_test_r2=r2_score(y_test, y_pred_rf_log_dollars)
rf_log_test_mae=mean_absolute_error(y_test, y_pred_rf_log_dollars)

The random forest is largely insensitive to whether purchasing power is modeled on its original or logarithmic scale. Fitting the model to the logged outcome and back transforming predictions gives an R² of 0.877, compared with 0.876 when the outcome is modeled directly. Because the log specification is also used for the primary linear model, the log target forest is used for the ablations and model comparisons that follow. The interpretability measures below, grouped permutation and partial dependence, are computed on the raw target forest, which reports directly in dollars.

In [53]:
feature_cols_no_exposure=[c for c in feature_cols_reg if c not in exposure_cols_model]
X_train_ne=X_train[feature_cols_no_exposure]
X_test_ne=X_test[feature_cols_no_exposure]

rf_no_exposure=RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf_no_exposure.fit(X_train_ne, y_train_log)
rf_no_exposure_r2=r2_score(y_test, np.exp(rf_no_exposure.predict(X_test_ne)))

Ablation measures how well the model compensates when a whole block of predictors is removed and it is refit from scratch. Removing the task groups reduces held out R² from 0.876 to 0.834, a decline of 0.042.

In [54]:
feature_cols_no_econ=[c for c in feature_cols_reg if c not in ["poverty_rate", "unemployment_rate"]]
X_train_ne2=X_train[feature_cols_no_econ]
X_test_ne2=X_test[feature_cols_no_econ]

rf_no_econ=RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf_no_econ.fit(X_train_ne2, y_train_log)
rf_no_econ_r2=r2_score(y_test, np.exp(rf_no_econ.predict(X_test_ne2)))

By comparison, a model with the task groups but without poverty and unemployment reaches an R² of 0.698. Task groups carry substantial explanatory information on their own, but the strongest performance comes from combining them with the economic controls. All three models use the same forest specification, logged outcome, and held out counties, and their performance is directly comparable.

#### Grouped permutation importance

To see how strongly the fitted model actually relies on the task groups, we permute the modeled task group proportions jointly and measure the resulting drop in R², leaving the fitted model itself untouched.

In [55]:
exposure_cols=exposure_cols_model  # the three non-reference task groups
rng=np.random.RandomState(42)

baseline_r2=r2_score(y_test, best_rf.predict(X_test))

def grouped_permutation_drop(cols, n_repeats=20):
    drops=[]
    for _ in range(n_repeats):
        X_perm=X_test.copy()
        shuffled_idx=rng.permutation(len(X_perm))
        X_perm[cols]=X_perm[cols].values[shuffled_idx]
        drops.append(baseline_r2-r2_score(y_test, best_rf.predict(X_perm)))
    return np.mean(drops)

joint_drop=grouped_permutation_drop(exposure_cols)

individual_drops={}
for col in exposure_cols:
    individual_drops[col]=grouped_permutation_drop([col])

year_drop=grouped_permutation_drop(year_cols_model)
state_drop=grouped_permutation_drop(state_cols_model)

Jointly permuting the task group proportions drops R² by 0.153, well beyond the 0.042 decline from removing them and refitting. The gap points to a difference between the two methods. When the task groups are dropped before training, the forest can recover part of their signal from correlated predictors such as poverty and time. Permutation instead disrupts that information after the model has already learned to rely on it, and no such recovery is possible. The task groups overlap with poverty, unemployment, and time while still adding something those three do not capture on their own.

Year produces a substantially larger permutation decline than state. The drop is 0.128 for year against 0.008 for state, conditional on the economic controls and task groups already in the model. Time carries far more of the model’s remaining predictive signal than geography does.

Among the three modeled proportions, non-routine manual shows the largest individual permutation drop. None of the three should be read as additive or independent contributions, since the task groups are compositional and mechanically related to one another. This ranking is consistent with the panel regression, where non-routine manual carries the steepest coefficient among the task groups.

### Neural network

The forest gained nothing on the log linear fit, which leaves open whether that ceiling belongs to the forest or to the data. A neural network is the next rung on the ladder and the check on that question. We keep it shallow, with three hidden layers and a single linear output unit for the continuous outcome.

In [56]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(42)
np.random.seed(42)

scaler_x=StandardScaler().fit(X_train)
X_train_s=scaler_x.transform(X_train)
X_test_s=scaler_x.transform(X_test)

scaler_y=StandardScaler().fit(y_train.values.reshape(-1, 1))
y_train_s=scaler_y.transform(y_train.values.reshape(-1, 1)).ravel()
y_test_s=scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

n_features=X_train.shape[1]
h1=int(round(n_features*1.5))
h2=max(int(round(h1*0.5)), 20)
h3=10

nn=models.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(h1, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(h2, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(h3, activation="relu"),
    layers.Dense(1, activation="linear"),  # regression head
])

nn.compile(optimizer="adam", loss="mse", metrics=["mae"])

es=callbacks.EarlyStopping(patience=15, restore_best_weights=True)

history=nn.fit(
    X_train_s, y_train_s,
    validation_split=0.2,
    epochs=300,
    batch_size=32,
    callbacks=[es],
    verbose=0,
)

y_pred_nn_s=nn.predict(X_test_s).ravel()
y_pred_nn=scaler_y.inverse_transform(y_pred_nn_s.reshape(-1, 1)).ravel()

nn_initial_test_r2=r2_score(y_test, y_pred_nn)
nn_initial_test_mae=mean_absolute_error(y_test, y_pred_nn)

nn_init_df=pd.DataFrame({
    "epoch": range(1, len(history.history["loss"])+1),
    "Train": history.history["loss"],
    "Validation": history.history["val_loss"],
}).melt(id_vars="epoch", var_name="series", value_name="loss")

Training loss falls steadily, while validation loss reverses direction partway through. That divergence is overfitting, and <a href="#fig-nn-curves" class="quarto-xref">Figure 16</a> panel A shows it alongside the regularized model in panel B. We respond with L2 weight regularization, heavier dropout, and a shorter early stopping patience.

In [57]:
from tensorflow.keras import regularizers

nn=models.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(h1, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(h2, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),
    layers.Dense(h3, activation="relu"),
    layers.Dense(1, activation="linear"),
])

nn.compile(optimizer="adam", loss="mse", metrics=["mae"])

es=callbacks.EarlyStopping(patience=8, restore_best_weights=True)

history=nn.fit(
    X_train_s, y_train_s,
    validation_split=0.2,
    epochs=300,
    batch_size=32,
    callbacks=[es],
    verbose=0,
)

y_pred_nn_s=nn.predict(X_test_s).ravel()
y_pred_nn=scaler_y.inverse_transform(y_pred_nn_s.reshape(-1, 1)).ravel()

nn_test_r2=r2_score(y_test, y_pred_nn)
nn_test_mae=mean_absolute_error(y_test, y_pred_nn)

nn_reg_df=pd.DataFrame({
    "epoch": range(1, len(history.history["loss"])+1),
    "Train": history.history["loss"],
    "Validation": history.history["val_loss"],
}).melt(id_vars="epoch", var_name="series", value_name="loss")

In [58]:
%%R -i nn_init_df -i nn_reg_df -w 10 -h 4.5 -u in -r 150 -b transparent
ymax <- max(c(nn_init_df$loss, nn_reg_df$loss)) * 1.05
# breaks run the whole axis rather than stopping at 0.8 and leaving the top of the
# validation curve untick-marked
y_breaks <- seq(0, ceiling(ymax*10)/10, by=0.1)

best_init <- nn_init_df[nn_init_df$series=="Validation",]
best_init_epoch <- best_init$epoch[which.min(best_init$loss)]
best_init_loss <- min(best_init$loss)
best_reg <- nn_reg_df[nn_reg_df$series=="Validation",]
best_reg_epoch <- best_reg$epoch[which.min(best_reg$loss)]
best_reg_loss <- min(best_reg$loss)

label_df1 <- nn_init_df[nn_init_df$epoch==max(nn_init_df$epoch),]
label_df2 <- nn_reg_df[nn_reg_df$epoch==max(nn_reg_df$epoch),]
label_nudge <- ymax*0.018
label_df1$label_y <- label_df1$loss + ifelse(label_df1$series=="Train", -label_nudge, label_nudge)
label_df2$label_y <- label_df2$loss + ifelse(label_df2$series=="Train", -label_nudge, label_nudge)

p1 <- ggplot(nn_init_df, aes(x=epoch, y=loss, color=series)) +
  geom_segment(aes(x=best_init_epoch, xend=best_init_epoch, y=0, yend=ymax),
               inherit.aes=FALSE, linetype="dashed", linewidth=0.35, color="black") +
  geom_line(linewidth=0.8) +
  geom_point(data=best_init[best_init$epoch==best_init_epoch,], aes(x=epoch, y=loss),
             inherit.aes=FALSE, color=ORANGE, size=2.2) +
  geom_label(data=label_df1, aes(label=series, y=label_y), hjust=-0.1, size=3.4, fontface="bold",
             fill=NA, linewidth=0, label.padding=unit(0.08, "lines"), show.legend=FALSE) +
  # the minimum sits near the left edge here, so the label reads rightward off the
  # marker instead of running off the panel and into the y axis title. Nudged
  # further right and up so the now transparent label clears the validation curve.
  annotate("label", x=best_init_epoch + 0.6, y=best_init_loss + ymax*0.26,
           label=paste0("Minimum validation loss\nepoch ", best_init_epoch),
           hjust=0, size=2.9, color="grey35", fill=NA, linewidth=0,
           label.padding=unit(0.12, "lines")) +
  scale_color_manual(values=c(Train=BLUE, Validation=ORANGE), guide="none") +
  scale_y_continuous(limits=c(0, ymax), breaks=y_breaks) +
  # limits start at 0 so the axis carries a 0 tick, even though epochs begin at 1
  scale_x_continuous(breaks=scales::pretty_breaks(n=10), limits=c(0, NA),
                     expand=expansion(mult=c(0.02, 0.28))) +
  labs(x="Epoch", y="Mean Squared Error (MSE)", title="Initial Model",
       subtitle="Patience 15")

p2 <- ggplot(nn_reg_df, aes(x=epoch, y=loss, color=series)) +
  geom_segment(aes(x=best_reg_epoch, xend=best_reg_epoch, y=0, yend=ymax),
               inherit.aes=FALSE, linetype="dashed", linewidth=0.35, color="black") +
  geom_line(linewidth=0.8) +
  geom_point(data=best_reg[best_reg$epoch==best_reg_epoch,], aes(x=epoch, y=loss),
             inherit.aes=FALSE, color=ORANGE, size=2.2) +
  geom_label(data=label_df2, aes(label=series, y=label_y), hjust=-0.1, size=3.4, fontface="bold",
             fill=NA, linewidth=0, label.padding=unit(0.08, "lines"), show.legend=FALSE) +
  # reads rightward off the dashed line and sits well above both curves, matching
  # panel A; the transparent fill means it must clear the lines rather than mask them
  annotate("label", x=best_reg_epoch + 0.8, y=best_reg_loss + ymax*0.45,
           label=paste0("Minimum validation loss\nepoch ", best_reg_epoch),
           hjust=0, size=2.9, color="grey35", fill=NA, linewidth=0,
           label.padding=unit(0.12, "lines")) +
  scale_color_manual(values=c(Train=BLUE, Validation=ORANGE), guide="none") +
  scale_y_continuous(limits=c(0, ymax), breaks=y_breaks) +
  scale_x_continuous(breaks=scales::pretty_breaks(n=10), expand=expansion(mult=c(0.02, 0.28))) +
  labs(x="Epoch", y="Mean Squared Error (MSE)", title="Regularized Model",
       subtitle="L2 Regularization, Patience 8")

p1 + p2 +
  plot_annotation(title="Training and Validation Loss, Initial vs. Regularized Network",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B"))) &
  theme(panel.grid.major.y=element_blank(), panel.grid.major.x=element_blank(),
        axis.text.y=element_text(), axis.text.x=element_text()) &
  theme(panel.grid.major=element_blank(), panel.grid.minor=element_blank())

R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1:   
R callback write-console: In geom_segment(aes(x = best_init_epoch, xend = best_init_epoch,  :  
R callback write-console: 
   
R callback write-console:  All aesthetics have length 1, but the data has 36 rows.
ℹ Please consider using `annotate()` or provide this layer with data containing
  a single row.
  
R callback write-console: 2:   
R callback write-console: In geom_segment(aes(x = best_reg_epoch, xend = best_reg_epoch, y = 0,  :  
R callback write-console: 
   
R callback write-console:  All aesthetics have length 1, but the data has 98 rows.
ℹ Please consider using `annotate()` or provide this layer with data containing
  a single row.
  

The gap narrows after regularization, but the network still does not beat the random forest or the log target linear model, both of which are far cheaper to fit and tune. This project is built to explain the relationship between task groups and purchasing power rather than to maximize predictive accuracy. We stop tuning here and carry the linear and forest models forward as the primary results.

### Cross validation

Having compared explained variance across the three models, we cross validate to check that the single split result generalizes.

In [59]:
from sklearn.model_selection import GroupKFold

group_kfold_cv=GroupKFold(n_splits=5)
cv_results={
    "Ordinary Least Squares": [],
    "Random Forest": [],
}

y_log=np.log(y)  # full data log target, same idea as y_train_log/y_test_log

for fold, (tr_idx, val_idx) in enumerate(group_kfold_cv.split(X_ml, y, groups)):
    X_tr, X_val=X_ml.iloc[tr_idx], X_ml.iloc[val_idx]
    y_tr, y_val=y.iloc[tr_idx], y.iloc[val_idx]
    y_tr_log=y_log.iloc[tr_idx]

    X_tr_sm=sm.add_constant(X_tr)
    X_val_sm=sm.add_constant(X_val, has_constant="add")
    ols_fold_log=sm.OLS(y_tr_log, X_tr_sm).fit()
    pred_ols=np.exp(ols_fold_log.predict(X_val_sm))  # back transformed, dollar scale
    cv_results["Ordinary Least Squares"].append(r2_score(y_val, pred_ols))

    rf_fold=RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
    rf_fold.fit(X_tr, y_tr_log)
    pred_rf=np.exp(rf_fold.predict(X_val))
    cv_results["Random Forest"].append(r2_score(y_val, pred_rf))

cv_folds=pd.DataFrame([
    {"model": m, "fold": i+1, "r2": s}
    for m, scores in cv_results.items()
    for i, s in enumerate(scores)
])

cv_summary=pd.DataFrame({m: {"Mean R²": np.mean(s), "SD": np.std(s)}
                         for m, s in cv_results.items()}).T.reset_index()
cv_summary.columns=["Model","Mean R²","SD"]

In [60]:
style_table(GT(cv_summary)
  .fmt_number(columns=["Mean R²","SD"], decimals=4)
  .cols_align(align="right", columns=["Mean R²","SD"])
  .cols_width(cases={"Model": "130px", "Mean R²": "90px", "SD": "90px"})
  .tab_source_note("Both models are fit on the log target and scored on the dollar scale after back transformation."))

/opt/anaconda3/envs/capstone/lib/python3.12/site-packages/great_tables/_render_checks.py:37: RenderWarning: Rendering table with .cols_width() in Quarto may result in unexpected behavior. This is because Quarto performs custom table processing. Either use all percentage widths, or set .tab_options(quarto_disable_processing=True) to disable Quarto table processing.
  warnings.warn(

<a href="#tbl-cv-summary" class="quarto-xref">Table 6</a> shows the two models performing at the same level across five folds. The difference between their means is smaller than the variation across the folds themselves, so the single split result above is not an artifact of one particular split. The neural network is not included, since it is not carried forward. <a href="#sec-results" class="quarto-xref">Section 5</a> reports the model comparison across held out R² and mean absolute error.

## Summary

The panel regression, with standard errors clustered by county, is the model used for inference. It shows that routine intensive work is associated with lower purchasing power even after controlling for poverty, unemployment, population, and year. Its log respecification resolves the residual violations that make the raw target version untrustworthy for that purpose. The variance decomposition adds that the task group differences behind that association are durable features of a place rather than transient ones.

The predictive models answer a different question, which is how much of purchasing power can be explained at all, and by what. On held out data the random forest and the log target linear model perform comparably, and the curvature in the relationship adds little once poverty, unemployment, time, and geography are already in the model. The neural network improves on neither, and its added complexity is not justified by this feature set.

Across both approaches the task groups carry independent explanatory power, while poverty rate remains the dominant single factor. That ordering is the finding this analysis is built to support, and it is what <a href="#sec-results" class="quarto-xref">Section 5</a> carries forward.

# Results

In [61]:
# inbound from _04: reg, df, model, model_log, X_ml, X_test, y_test, y_pred_rf,
# y_pred_rf_log_dollars, best_rf, perm_imp, joint_drop, year_drop, state_drop,
# cv_folds, engine, ols_log_test_r2, rf_log_test_r2, nn_test_r2,
# ols_log_test_mae, rf_log_test_mae, nn_test_mae
import pandas as pd
import numpy as np
import statsmodels.api as sm
from great_tables import GT, html

res_cols=["routine_cognitive_share","routine_manual_share","non_routine_manual_share",
          "poverty_rate","unemployment_rate","log_population"]

mean_pp=float(reg["affordability_salary"].mean())
group_terms=["non_routine_manual_share","routine_cognitive_share","routine_manual_share"]
group_names=["Non-Routine Manual","Routine Cognitive","Routine Manual"]

The analysis supports four findings. A county’s task groups are associated with its purchasing power, and the association survives controls for poverty, unemployment, population, and year. The log specification captures most of the structure in the relationship, and the flexible models do not improve on it, which is what supports reading the relationship as proportional rather than fixed in dollars. Poverty rate remains the single strongest correlate of purchasing power throughout, yet the task groups add signal beyond it. Finally, the task group differences behind the association are durable features of places, and so are the purchasing power gaps they carry. The subsections below work through these findings, and through the geography and the model behavior behind them.

## The association

In [62]:
# log specification coefficients (model_log fit in _04, cluster robust), expressed as
# dollars at the panel mean per one percentage point shift
coef_rows=[]
for term, name in zip(group_terms, group_names):
    b=model_log.params[term]
    se=model_log.bse[term]
    est=(np.exp(b*0.01)-1)*mean_pp
    lo=(np.exp((b-1.96*se)*0.01)-1)*mean_pp
    hi=(np.exp((b+1.96*se)*0.01)-1)*mean_pp
    coef_rows.append({"group":name, "est":est, "lo":lo, "hi":hi})
coef_df=pd.DataFrame(coef_rows)

The log specification of the panel regression is the model we use for inference, for the diagnostic reasons set out in <a href="#sec-analysis" class="quarto-xref">Section 4</a>. A one percentage point shift toward non-routine manual work is associated with roughly \$846 less in purchasing power than an equivalent share of non-routine cognitive work, the reference group, with a 95 percent interval of plus or minus \$38 at the panel mean (<a href="#fig-coefplot" class="quarto-xref">Figure 17</a>). Routine cognitive and routine manual shifts carry smaller but clearly negative associations, in the same ordering the level specification reported in <a href="#sec-analysis" class="quarto-xref">Section 4</a> produces. For scale, each additional percentage point of poverty rate is associated with roughly 2.9 percent lower purchasing power, a larger proportional difference than any single task group carries.

In [63]:
%%R -i coef_df -w 8 -h 3.5 -u in -r 150 -b transparent
dollar_axis <- function(v) ifelse(is.na(v), "", ifelse(v == 0, "$0",
  sprintf("%s$%s", ifelse(v < 0, "-", ""), formatC(abs(v), format="d", big.mark=","))))
coef_df$group <- factor(coef_df$group,
    levels=rev(c("Non-Routine Manual","Routine Cognitive","Routine Manual")))

ggplot(coef_df, aes(x=est, y=group, color=group)) +
  geom_vline(xintercept=0, linetype="dashed", linewidth=0.9, color="#0B0B0B") +
  geom_errorbar(aes(xmin=lo, xmax=hi), orientation="y", width=0.15) +
  geom_point(size=2.6) +
  geom_text(aes(label=dollar_axis(est)), vjust=-1.4, size=3.3, fontface="bold", show.legend=FALSE) +
  scale_color_manual(values=TASK_COLORS, guide="none") +
  # denser ticks on the same dollar scale; horizontal gridlines dropped because each
  # row is read across to the x scale, not compared vertically
  scale_x_continuous(labels=dollar_axis, breaks=scales::pretty_breaks(n=10),
                     expand=expansion(mult=c(0.1, 0.15))) +
  labs(title="Task Group Coefficients, Log Specification",
       x="Purchasing Power Change ($ per 1 pp)", y=NULL) +
  theme(panel.grid.major.y=element_blank())

In the level specification, poverty rate carries the largest and most precisely estimated coefficient, while unemployment rate and log population are smaller but still distinguishable from zero at the 0.05 level (<a href="#tbl-panel-coefs" class="quarto-xref">Table 7</a>). The table also reports the model’s overall F-test.

In [64]:
level_coef_terms=["routine_cognitive_share","routine_manual_share","non_routine_manual_share",
                   "poverty_rate","unemployment_rate","log_population"]
level_coef_labels={
    "routine_cognitive_share":"Routine Cognitive",
    "routine_manual_share":"Routine Manual",
    "non_routine_manual_share":"Non-Routine Manual",
    "poverty_rate":"Poverty Rate",
    "unemployment_rate":"Unemployment Rate",
    "log_population":"Log Population",
}
level_coef_df=pd.DataFrame({
    "Variable": [level_coef_labels[t] for t in level_coef_terms],
    "Coefficient": [model.params[t] for t in level_coef_terms],
    "Std. Error": [model.bse[t] for t in level_coef_terms],
    "t": [model.tvalues[t] for t in level_coef_terms],
    "p-value": [model.pvalues[t] for t in level_coef_terms],
})
level_f_note=(f"F({int(model.df_model)}, {int(model.df_resid)}) = {model.fvalue:.1f}, "
              f"p {'< 0.001' if model.f_pvalue < 0.001 else f'= {model.f_pvalue:.3f}'}; "
              f"n = {int(model.nobs):,}; R² = {model.rsquared:.3f}. "
              "Standard errors clustered by county.")

In [65]:
style_table(GT(level_coef_df)
  .fmt_currency(columns="Coefficient", decimals=0)
  .fmt_number(columns="Std. Error", decimals=0, use_seps=True)
  .fmt_number(columns="t", decimals=2)
  .fmt_number(columns="p-value", decimals=3)
  .tab_source_note(level_f_note))

## Not poverty in disguise

A natural objection is that the task groups merely proxy for poverty, since poor counties hold more routine work. <a href="#fig-baseline-maps" class="quarto-xref">Figure 18</a> shows the two conventional distress measures side by side, mean poverty rate (Panel A) and mean unemployment rate (Panel B) by county across the study period. Both run highest across the rural South, the Mississippi Delta, and pockets of the Southwest border region. That is the same broad area where <a href="#fig-afford-map" class="quarto-xref">Figure 23</a> shows purchasing power running lowest, which is exactly why the objection is worth taking seriously. Unemployment (Panel B) carries a second concentration along the California coast and Central Valley that poverty (Panel A) does not share, evidence the two measures are not interchangeable.

In [66]:
import geopandas as gpd

map_baseline=pd.read_sql("""
    select county_fips, avg(poverty_rate) as poverty_rate, avg(unemployment_rate) as unemployment_rate
    from county_baseline
    group by county_fips
""", engine)
map_baseline['fips']=map_baseline['county_fips'].astype(str).str.zfill(5)

gdf_baseline=gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip")
gdf_baseline=gdf_baseline.rename(columns={'GEOID':'fips'})
merged_baseline=gdf_baseline.merge(map_baseline[['fips','poverty_rate','unemployment_rate']], on='fips', how='left')
merged_baseline=merged_baseline[~merged_baseline['STATEFP'].isin(['02','15','60','66','69','72','78'])]
os.makedirs("output", exist_ok=True)
merged_baseline[['fips','poverty_rate','unemployment_rate','geometry']].to_file("output/baseline_maps.geojson", driver="GeoJSON")

pov_vmin=float(merged_baseline['poverty_rate'].quantile(0.02))
pov_vmax=float(merged_baseline['poverty_rate'].quantile(0.98))
unemp_vmin=float(merged_baseline['unemployment_rate'].quantile(0.02))
unemp_vmax=float(merged_baseline['unemployment_rate'].quantile(0.98))

In [67]:
%%R -i pov_vmin -i pov_vmax -i unemp_vmin -i unemp_vmax -w 11 -h 5.5 -u in -r 150 -b transparent
suppressMessages(library(sf))

baseline_sf <- st_read("output/baseline_maps.geojson", quiet=TRUE)

p1 <- ggplot(baseline_sf) +
  geom_sf(aes(fill=poverty_rate), color="white", linewidth=0.05) +
  scale_fill_distiller(palette="YlOrBr", direction=1, limits=c(pov_vmin, pov_vmax), oob=scales::squish,
                        na.value="#eeeeee", name="Poverty Rate (%)") +
  coord_sf(crs=st_crs(5070)) +
  theme_void() +
  theme(legend.position="bottom", legend.key.width=unit(1.2, "cm"),
        plot.title=element_text(face="bold", size=11, hjust=0.5),
        plot.background=element_rect(fill=NA, color=NA)) +
  labs(title="Poverty Rate")

p2 <- ggplot(baseline_sf) +
  geom_sf(aes(fill=unemployment_rate), color="white", linewidth=0.05) +
  scale_fill_distiller(palette="YlOrBr", direction=1, limits=c(unemp_vmin, unemp_vmax), oob=scales::squish,
                        na.value="#eeeeee", name="Unemployment Rate (%)") +
  coord_sf(crs=st_crs(5070)) +
  theme_void() +
  theme(legend.position="bottom", legend.key.width=unit(1.2, "cm"),
        plot.title=element_text(face="bold", size=11, hjust=0.5),
        plot.background=element_rect(fill=NA, color=NA)) +
  labs(title="Unemployment Rate")

p1 + p2 +
  plot_annotation(title="Two Conventional Distress Measures, by County",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B")))

Fitting the log panel regression twice on the same rows tests the objection directly (<a href="#fig-coefcompare" class="quarto-xref">Figure 19</a>). The first specification includes only task groups and year indicators; the second adds poverty rate, unemployment rate, and log population, the specification reported in <a href="#sec-analysis" class="quarto-xref">Section 4</a>. Non-routine manual and routine manual both shrink substantially when the controls enter, which is what we would expect, since poverty absorbs shared variation. Routine cognitive barely moves, at roughly −1.1 percent either way, evidence its association runs through something other than the poverty and unemployment channel the controls capture. All three remain negative and precisely estimated after the controls enter. The task group association is smaller than it first appears for two of the three groups and clearly present for all of them.

In [68]:
share_terms=["routine_cognitive_share","routine_manual_share","non_routine_manual_share"]

X_nocontrols=pd.concat([reg[share_terms],
                        pd.get_dummies(reg["year"], prefix="year", drop_first=True).astype(float)], axis=1)
X_nocontrols=sm.add_constant(X_nocontrols)
model_nc_log=sm.OLS(reg["actual_log"], X_nocontrols).fit(
    cov_type="cluster", cov_kwds={"groups": reg["county_fips"]})

def pct_ci(m, term):
    b=m.params[term]
    se=m.bse[term]
    est=(np.exp(b*0.01)-1)*100
    lo=(np.exp((b-1.96*se)*0.01)-1)*100
    hi=(np.exp((b+1.96*se)*0.01)-1)*100
    return est, lo, hi

compare_rows=[]
for term, name in zip(share_terms, ["Routine Cognitive","Routine Manual","Non-Routine Manual"]):
    for m, spec in [(model_nc_log, "Baseline"), (model_log, "With controls")]:
        est, lo, hi = pct_ci(m, term)
        compare_rows.append({"group":name, "spec":spec, "est":est, "lo":lo, "hi":hi})
compare_df=pd.DataFrame(compare_rows)

compare_wide=compare_df.pivot(index="group", columns="spec", values="est").reset_index()
compare_wide.columns=["group","est_baseline","est_controls"]

In [69]:
%%R -i compare_df -w 9 -h 4.5 -u in -r 150 -b transparent
compare_df$group <- factor(compare_df$group,
    levels=rev(c("Non-Routine Manual","Routine Cognitive","Routine Manual")))
compare_df$spec <- factor(compare_df$spec,
    levels=c("Baseline","With controls"))

# the two specifications sit on their own y offsets within each task group, so the
# estimates and their intervals never overlap even where the two barely differ
dodge <- position_dodge(width=0.55)
lab_pad <- diff(range(c(compare_df$lo, compare_df$hi))) * 0.02

ggplot(compare_df, aes(x=est, xmin=lo, xmax=hi, y=group, color=spec, shape=spec)) +
  geom_vline(xintercept=0, linetype="dashed", linewidth=0.9, color="#0B0B0B") +
  geom_errorbar(orientation="y", position=dodge, width=0.18, linewidth=0.6) +
  geom_point(position=dodge, fill="white", size=3.2, stroke=1.0) +
  geom_text(aes(x=hi + lab_pad, label=sprintf("%.1f%%", est)), position=dodge,
            hjust=0, size=3.2, fontface="bold", show.legend=FALSE) +
  scale_color_manual(values=c("Baseline"=ORANGE, "With controls"=GREEN)) +
  scale_shape_manual(values=c("Baseline"=21, "With controls"=19)) +
  # the percent sign moves onto the tick values, so the axis title stays clean
  scale_x_continuous(breaks=scales::pretty_breaks(n=10),
                     labels=function(v) sprintf("%g%%", v),
                     expand=expansion(mult=c(0.04, 0.10))) +
  scale_y_discrete(expand=expansion(add=0.6)) +
  labs(title="Task Group Coefficients Before and After Controls",
       x="Purchasing Power Change", y=NULL, color=NULL, shape=NULL) +
  theme(legend.position="bottom", panel.grid.major.y=element_blank())

## What carries the signal

Two measurements say how much the task groups contribute. The first removes them and refits, and <a href="#fig-ablation" class="quarto-xref">Figure 20</a> shows what the random forest loses in held out R² when each block of features is taken away. Removing poverty and unemployment costs 0.178, more than four times the 0.042 lost by removing the task groups. The economic controls therefore carry more explanatory information, but the task groups still contribute beyond them.

In [70]:
ablation_df=pd.DataFrame({
    "model": ["Task Groups","Economic Controls"],
    "r2": [0.834, 0.698],
})
ablation_df["full_r2"]=0.876
ablation_df["drop"]=ablation_df["full_r2"]-ablation_df["r2"]

In [71]:
%%R -i ablation_df -w 7 -h 3 -u in -r 150 -b transparent
ablation_df$model <- factor(ablation_df$model, levels=c("Task Groups","Economic Controls"))

ggplot(ablation_df, aes(x=drop, y=model, fill=model)) +
  geom_col(width=0.55) +
  geom_text(aes(label=sprintf("%.3f", drop)), hjust=-0.25, size=4.0, fontface="bold", color="#252525") +
  scale_fill_manual(values=c("Task Groups"=GREEN, "Economic Controls"=ORANGE), guide="none") +
  scale_x_continuous(limits=c(0, 0.20), breaks=seq(0, 0.20, 0.05),
                     labels=function(v) sprintf("%.2f", v), expand=c(0,0)) +
  labs(title="Held Out R² Loss From Feature Block Removal",
       x="R² Loss", y=NULL) +
  theme(panel.grid.major.y=element_blank())

The second measurement leaves the model intact and destroys the information instead. Permuting each feature or block in turn ranks what the random forest relies on, using the grouped permutation approach from <a href="#sec-analysis" class="quarto-xref">Section 4</a> so that indicator blocks and the task groups are each scored as a unit (<a href="#fig-importance" class="quarto-xref">Figure 21</a>). These two interpretability figures are computed on the raw target forest rather than the log target one, since the two score within 0.001 of each other on held out counties. We report the raw target version because it reads directly in dollars. In that ranking, poverty rate carries the largest single drop, consistent with its dominance in the regression. The modeled task group proportions together come next at 0.153, ahead of the year block at 0.128. State adds almost nothing, 0.008, once everything else is present.

The two measurements disagree in magnitude, and the reason is instructive. Permuting the task groups costs 0.153, while removing them entirely and refitting costs only 0.042. The gap exists because a refitted model can lean harder on poverty, unemployment, and time to recover much of what the task groups were carrying. The permutation figure therefore measures how much the fitted model uses the task groups, and the ablation measures how much of that is unique to them. Both measurements agree that the task groups function together as a set, which is what parts of a whole should do.

In [72]:
pov_idx=list(X_test.columns).index("poverty_rate")
unemp_idx=list(X_test.columns).index("unemployment_rate")

importance_df=pd.DataFrame({
    "feature": ["Poverty Rate","Task Groups","Year",
                "Unemployment Rate","State"],
    "block": ["Economic Controls","Task Groups","Structural Indicators",
              "Economic Controls","Structural Indicators"],
    "drop": [perm_imp.importances_mean[pov_idx], joint_drop, year_drop,
             perm_imp.importances_mean[unemp_idx], state_drop],
}).sort_values("drop")

In [73]:
%%R -i importance_df -w 8 -h 3.5 -u in -r 150 -b transparent
importance_df$feature <- factor(importance_df$feature, levels=importance_df$feature)
importance_df$block <- factor(importance_df$block,
    levels=c("Task Groups","Economic Controls","Structural Indicators"))

ggplot(importance_df, aes(x=drop, y=feature, fill=block)) +
  geom_col(width=0.6) +
  geom_text(aes(label=sprintf("%.3f", drop)), hjust=-0.25, size=3.4, fontface="bold", color="#252525") +
  scale_fill_manual(values=c("Task Groups"=GREEN, "Economic Controls"=ORANGE,
                             "Structural Indicators"=GREY)) +
  # ticks every 0.05; ggplot keeps only those inside the existing range, so the
  # scale itself is unchanged
  scale_x_continuous(breaks=seq(0, 1, 0.05), labels=function(v) sprintf("%.2f", v),
                     expand=expansion(mult=c(0, 0.15))) +
  labs(title="Feature Importance by Grouped Permutation",
       x="Drop in Held Out R² When Permuted", y=NULL, fill=NULL) +
  theme(legend.position="bottom", panel.grid.major.y=element_blank())

Across the observed range of each non-reference task group, the forest’s partial dependence declines smoothly and near monotonically, without thresholds or reversals (<a href="#fig-pdp" class="quarto-xref">Figure 22</a>). That shape is what a proportional association implies, which is why the added flexibility bought nothing over the log linear fit. Because the four groups are parts of a whole, moving one while holding the others fixed implies a compensating change in the omitted reference group. These curves therefore describe the model’s behavior rather than a combination any county could occupy.

In [74]:
from sklearn.inspection import partial_dependence

pdp_frames=[]
for term, name in zip(["routine_cognitive_share","routine_manual_share","non_routine_manual_share"],
                      ["Routine Cognitive","Routine Manual","Non-Routine Manual"]):
    pd_res=partial_dependence(best_rf, X_test, [term], kind="average", grid_resolution=40)
    grid=pd_res["grid_values"][0] if "grid_values" in pd_res else pd_res["values"][0]
    pdp_frames.append(pd.DataFrame({"x":grid,
                                    "y":pd_res["average"][0],
                                    "group":name}))
pdp_df=pd.concat(pdp_frames, ignore_index=True)

In [75]:
%%R -i pdp_df -w 9.5 -h 3.8 -u in -r 150 -b transparent
pdp_levels <- c("Routine Cognitive","Routine Manual","Non-Routine Manual")
pdp_df$group <- factor(pdp_df$group, levels=pdp_levels)
PDP_COLORS <- TASK_COLORS[pdp_levels]

# the strip carries the group name alone, centred; the letter tag is drawn separately
# above the top of each panel's own y axis, so the two never share one string
tag_df <- data.frame(group=factor(pdp_levels, levels=pdp_levels),
                     tag=paste0(LETTERS[seq_along(pdp_levels)], ")"))

ggplot(pdp_df, aes(x=x, y=y, color=group)) +
  geom_line(linewidth=1.0) +
  geom_text(data=tag_df, aes(x=-Inf, y=Inf, label=tag), inherit.aes=FALSE,
            hjust=1.2, vjust=-0.45, fontface="bold", size=4.2, color="#0B0B0B") +
  # axes="all_y" repeats the y axis on every panel; the scale itself stays shared,
  # so the three declines remain comparable in magnitude
  facet_wrap(~group, ncol=3, axes="all_y") +
  coord_cartesian(clip="off") +
  scale_color_manual(values=PDP_COLORS, guide="none") +
  scale_y_continuous(labels=function(v) sprintf("$%s", formatC(round(v), format="d", big.mark=",")),
                     breaks=scales::pretty_breaks(n=7),
                     expand=expansion(mult=c(0.02, 0.02))) +
  scale_x_continuous(labels=function(v) v * 100, breaks=scales::pretty_breaks(n=6)) +
  labs(title="Partial Dependence on Each Task Group",
       x="Shift from Non-Routine Cognitive (Percentage Points)",
       y="Predicted Purchasing Power") +
  theme(panel.spacing=unit(1.2, "lines"), plot.margin=margin(16, 10, 8, 10))

## Where purchasing power is strained

In [76]:
import geopandas as gpd

map_all=pd.read_sql("""
    select county_fips, avg(affordability_salary) as purchasing_power
    from county_affordability
    group by county_fips
""", engine)
map_all['fips']=map_all['county_fips'].astype(str).str.zfill(5)

gdf_map=gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip")
gdf_map=gdf_map.rename(columns={'GEOID':'fips'})

afford_vmin=float(map_all['purchasing_power'].quantile(0.02))
afford_vmax=float(map_all['purchasing_power'].quantile(0.98))

merged_map=gdf_map.merge(map_all[['fips','purchasing_power']], on='fips', how='left')
merged_map=merged_map[~merged_map['STATEFP'].isin(['02','15','60','66','69','72','78'])]
os.makedirs("output", exist_ok=True)
merged_map[['fips','purchasing_power','geometry']].to_file("output/afford_map.geojson", driver="GeoJSON")

# dominant task group per county: whichever of the four groups holds the largest mean share
dominant_share_cols=["routine_cognitive_share","routine_manual_share",
                     "non_routine_cognitive_share","non_routine_manual_share"]
dominant_label_map={"routine_cognitive_share":"Routine Cognitive","routine_manual_share":"Routine Manual",
                    "non_routine_cognitive_share":"Non-Routine Cognitive","non_routine_manual_share":"Non-Routine Manual"}
dominant_df=pd.read_sql(f"""
    select cte.county_fips,
           avg(cte.routine_cognitive_share) as routine_cognitive_share,
           avg(cte.routine_manual_share) as routine_manual_share,
           avg(cte.non_routine_cognitive_share) as non_routine_cognitive_share,
           avg(cte.non_routine_manual_share) as non_routine_manual_share,
           avg(ca.affordability_salary) as purchasing_power
    from county_task_exposure cte
    join county_affordability ca on cte.county_fips=ca.county_fips and cte.year=ca.year
    group by cte.county_fips
""", engine)
dominant_df["dominant_group"]=dominant_df[dominant_share_cols].idxmax(axis=1).map(dominant_label_map)

dominant_n=dominant_df["dominant_group"].value_counts()
dominant_med=dominant_df.groupby("dominant_group")["purchasing_power"].median()

The association also has a clear geographic pattern. Purchasing power runs highest along the metropolitan Northeast corridor, across parts of the upper Midwest, and in pockets of the mountain West, while the lowest values concentrate across the rural South and the southern border region (<a href="#fig-afford-map" class="quarto-xref">Figure 23</a>). That map uses all counties with a purchasing power value rather than the analytical panel alone, since the outcome requires only income and a price parity. That geography is associated with local task groups. Counties are grouped here by whichever task group accounts for the most local employment. The 664 counties where non-routine cognitive work dominates reach a median of \$61,162. The comparable medians are \$46,657 across the 97 non-routine manual counties and \$50,368 across the 87 routine manual ones. The association the coefficients estimate therefore holds not only across the continuous range but also at the level of a county’s single largest task group.

In [77]:
%%R -i afford_vmin -i afford_vmax -w 9 -h 6 -u in -r 150 -b transparent
suppressMessages(library(sf))

afford_sf <- st_read("output/afford_map.geojson", quiet=TRUE)

DIVERGING_COLORS <- c("#d53e4f", "#fc8d59", "#fee08b", "#e6f598", "#99d594", "#3288bd")

ggplot(afford_sf) +
  geom_sf(aes(fill=purchasing_power), color="#FAFAF8", linewidth=0.05) +
  scale_fill_gradientn(colors=DIVERGING_COLORS, limits=c(afford_vmin, afford_vmax),
                        oob=scales::squish, na.value="#D9D9D6",
                        labels=label_dollar(scale=1e-3, suffix="k"),
                        name="Mean Purchasing Power") +
  coord_sf(crs=st_crs(5070), datum=NA) +
  guides(fill=guide_colorbar(direction="horizontal", title.position="top", title.hjust=0.5,
                             barwidth=unit(4.5,"cm"), barheight=unit(0.35,"cm"))) +
  labs(title="Where Purchasing Power Is Strained") +
  theme_void() +
  theme(legend.position="bottom",
        legend.title=element_text(size=10, face="bold", color="#333333"),
        legend.text=element_text(size=9, color="#555555"),
        plot.title=element_text(face="bold", size=13, hjust=0.5),
        plot.background=element_rect(fill=NA, color=NA))

## The relationship is proportional

Across the five grouped cross validation folds, the log target linear model and the random forest perform similarly relative to the variation observed, and neither pulls away from the other (<a href="#fig-modelcomp" class="quarto-xref">Figure 24</a>). This collects the ending of the complexity ladder climbed in <a href="#sec-analysis" class="quarto-xref">Section 4</a>. <a href="#tbl-modelmetrics" class="quarto-xref">Table 8</a> adds a further comparison to that picture, a single split that also includes the neural network. The network reaches a held out R² of 0.874 with a mean absolute error of \$3,994. That is no better than either simpler model. The ladder was built to detect exactly this, and finding no improvement at the most flexible rung is what establishes the log linear specification as the right stopping point.

In [78]:
cv_means=cv_folds.groupby("model", as_index=False)["r2"].mean().rename(columns={"r2":"mean_r2"})

In [79]:
%%R -i cv_folds -i cv_means -w 8 -h 4 -u in -r 150 -b transparent
cv_means$mean_label <- sprintf("%.3f", cv_means$mean_r2)
cv_means$label_x <- ifelse(cv_means$model=="Ordinary Least Squares", 0.74, 2.26)

# evenly spread each fold's point off the shared category center, so the five folds
# per model don't stack into one vertical clump; the y values (and axis scale) are
# untouched, only the x position within each category changes
cv_folds$model_num <- as.numeric(factor(cv_folds$model, levels=c("Ordinary Least Squares","Random Forest")))
fold_ids <- sort(unique(cv_folds$fold))
fold_offset <- setNames(seq(-0.15, 0.15, length.out=length(fold_ids)), fold_ids)
cv_folds$x_spread <- cv_folds$model_num + fold_offset[as.character(cv_folds$fold)]
cv_means$model_num <- as.numeric(factor(cv_means$model, levels=c("Ordinary Least Squares","Random Forest")))

ggplot(cv_folds, aes(x=x_spread, y=r2, color=model)) +
  geom_line(aes(x=x_spread, y=r2, group=fold), inherit.aes=FALSE, color="grey75", linewidth=0.5) +
  geom_point(size=2.6) +
  geom_crossbar(data=cv_means, aes(x=model_num, y=mean_r2, ymin=mean_r2, ymax=mean_r2),
                width=0.35, linewidth=0.5, color="grey30", inherit.aes=FALSE) +
  geom_text(data=cv_means, aes(x=label_x, y=mean_r2, label=mean_label, color=model),
            size=3.4, fontface="bold", vjust=0.5, show.legend=FALSE) +
  scale_color_manual(values=c("Ordinary Least Squares"=BLUE,
                              "Random Forest"=PINK)) +
  scale_x_continuous(breaks=c(1,2), labels=c("Ordinary Least Squares","Random Forest"),
                      expand=expansion(add=0.6)) +
  # zoomed to the band the folds occupy. coord_cartesian rather than limits= so no
  # fold is dropped; note this magnifies a mean gap of roughly 0.006, which the
  # caption and surrounding prose both state in numbers rather than leaving to the eye
  scale_y_continuous(breaks=seq(0.875, 0.975, 0.025)) +
  coord_cartesian(ylim=c(0.875, 0.975)) +
  labs(title="Held Out R² by Fold, Linear Model vs. Random Forest",
       x=NULL, y="Held Out R²") +
  guides(color="none")

In [80]:
metrics_df=pd.DataFrame({
    "Model": ["Ordinary Least Squares","Random Forest","Neural Network"],
    "R²": [ols_log_test_r2, rf_log_test_r2, nn_test_r2],
    "Mean Absolute Error": [ols_log_test_mae, rf_log_test_mae, nn_test_mae],
})

style_table(GT(metrics_df)
  .fmt_number(columns="R²", decimals=3)
  .fmt_currency(columns="Mean Absolute Error", decimals=0)
  .tab_source_note("Ordinary Least Squares and Random Forest are fit on the log target and scored on the dollar scale after back transformation.")
  .tab_source_note("Neural network results are from this single split only; the five fold grouped cross validation table in the analysis section reports cross validated results for the other two models."))

This is the ladder’s stopping rule working as designed. A random forest can represent any interaction or curvature the data contain, yet it finds nothing beyond what the log transformation already captured. The binned comparison in <a href="#fig-binned" class="quarto-xref">Figure 25</a> confirms the same conclusion, this time in the units of the outcome itself. The observed relationship is not monotonic across these bins. Purchasing power rises with the routine cognitive value through the lower part of its observed range, peaks near 20 percent, and then declines as the value continues to rise. The bins and the partial dependence curves in <a href="#fig-pdp" class="quarto-xref">Figure 22</a> are not in conflict, since the curves hold the other three groups fixed. In these bins the other groups move together with routine cognitive, because the four are parts of a whole, and the bins therefore trace what counties actually look like rather than the model’s response to one group in isolation. The forest’s predictions track this same shape closely rather than discovering a different pattern of their own, which is further evidence that the flexible model adds little beyond what the log linear specification already captures.

In [81]:
binned_ml=pd.DataFrame({
    "rc": X_test["routine_cognitive_share"].values,
    "Actual": y_test.values,
    "Random Forest": y_pred_rf_log_dollars,
})
binned_ml["bin"]=pd.qcut(binned_ml["rc"], 10, labels=False)
binned_means=binned_ml.groupby("bin")[["rc","Actual","Random Forest"]].mean().reset_index(drop=True)
binned_means=binned_means.rename(columns={"Actual":"Observed","Random Forest":"Predicted"})
binned_long=binned_means.melt(id_vars="rc", value_vars=["Observed","Predicted"],
                              var_name="series", value_name="pp")

In [82]:
%%R -i binned_long -i rf_log_test_r2 -i rf_log_test_mae -w 9 -h 4.5 -u in -r 150 -b transparent
OBSERVED <- "#252525"
binned_long$rc_pct <- binned_long$rc*100
label_pts <- binned_long[binned_long$rc==max(binned_long$rc),]
rf_stat_label <- sprintf("R² = %.3f\nMAE = $%s",
                          rf_log_test_r2, formatC(round(rf_log_test_mae), format="d", big.mark=","))

ggplot(binned_long, aes(x=rc_pct, y=pp, color=series)) +
  annotate("text", x=max(binned_long$rc_pct), y=max(binned_long$pp)*0.975,
           label=rf_stat_label, hjust=1, size=2.9, color="#52514E") +
  geom_line(aes(linewidth=series)) +
  geom_point(aes(size=series)) +
  geom_text(data=label_pts, aes(label=series), hjust=-0.15, size=3.4, fontface="bold", show.legend=FALSE) +
  scale_color_manual(values=c("Observed"=OBSERVED, "Predicted"=BLUE), guide="none") +
  scale_linewidth_manual(values=c("Observed"=1.0, "Predicted"=0.8), guide="none") +
  scale_size_manual(values=c("Observed"=1.8, "Predicted"=1.5), guide="none") +
  scale_x_continuous(labels=function(v) sprintf("%d%%", round(v)),
                      breaks=scales::pretty_breaks(n=8),
                      expand=expansion(mult=c(0.02, 0.28))) +
  scale_y_continuous(labels=function(v) sprintf("$%s", formatC(round(v), format="d", big.mark=",")),
                     expand=expansion(mult=c(0.02, 0.02))) +
  labs(title="Observed and Predicted Purchasing Power by Routine Cognitive Value",
       x="Routine Cognitive Value", y="Purchasing Power") +
  theme(panel.grid.major=element_blank())

## The differences are durable

The final question is whether these associations describe a stable feature of places or a moment in time. Three pieces of evidence say stable. <a href="#fig-taskarea" class="quarto-xref">Figure 26</a> shows the panel average of each task group by year, which moves little across fifteen years apart from the 2009 to 2010 step produced by the Census occupation coding change described in <a href="#sec-data" class="quarto-xref">Section 3</a>.

In [83]:
area_df=(df.groupby("year")[["routine_cognitive_share","routine_manual_share",
                             "non_routine_cognitive_share","non_routine_manual_share"]]
           .mean().reset_index()
           .melt(id_vars="year", var_name="group", value_name="share"))
area_names={"routine_cognitive_share":"Routine Cognitive",
            "routine_manual_share":"Routine Manual",
            "non_routine_cognitive_share":"Non-Routine Cognitive",
            "non_routine_manual_share":"Non-Routine Manual"}
area_df["group"]=area_df["group"].map(area_names)
area_df["period"]=np.where(area_df["year"]<=2019, "pre", "post")

# narrow, deliberate gap right at the missing 2020 year, rather than the
# full two year blank span a plain pre/2021 split would otherwise leave
gap_rows=[]
for grp in area_df["group"].unique():
    before=area_df.loc[(area_df["group"]==grp) & (area_df["year"]==2019), "share"].iloc[0]
    after=area_df.loc[(area_df["group"]==grp) & (area_df["year"]==2021), "share"].iloc[0]
    gap_rows.append({"year":2019.9, "group":grp, "share":before+(after-before)*0.45, "period":"pre"})
    gap_rows.append({"year":2020.1, "group":grp, "share":before+(after-before)*0.55, "period":"post"})
area_df=pd.concat([area_df, pd.DataFrame(gap_rows)], ignore_index=True).sort_values("year").reset_index(drop=True)

In [84]:
%%R -i area_df -w 9.5 -h 4.7 -u in -r 150 -b transparent
TASKAREA_COLORS <- TASK_COLORS
group_levels <- c("Non-Routine Cognitive","Non-Routine Manual","Routine Cognitive","Routine Manual")
area_df$group <- factor(area_df$group, levels=group_levels)

pre_df  <- area_df[area_df$period=="pre",]
post_df <- area_df[area_df$period=="post",]

label_year <- 2016
label_x <- 2016.5  # drawn between the 2016 and 2017 ticks rather than on top of either
label_df <- area_df[area_df$year==label_year,]
label_df <- label_df[match(group_levels, label_df$group),]
label_df$cum <- cumsum(label_df$share)
label_df$mid <- label_df$cum - label_df$share/2
# Non-Routine Cognitive and Routine Cognitive are the light members of their hue
# families, so white labels lose contrast there; the two manual groups stay dark.
label_df$text_color <- ifelse(label_df$group %in% c("Non-Routine Manual","Routine Manual"),
                               "white", "#0B0B0B")

ggplot(area_df, aes(x=year, y=share, fill=group)) +
  annotate("rect", xmin=2019.9, xmax=2020.1, ymin=0, ymax=1, fill="grey60", alpha=0.35) +
  geom_area(data=pre_df, stat="identity", position=position_stack(reverse=TRUE), linewidth=0) +
  geom_area(data=post_df, stat="identity", position=position_stack(reverse=TRUE), linewidth=0) +
  geom_text(data=label_df, aes(x=label_x, y=mid, label=group, color=text_color),
            hjust=0.5, size=3.3, fontface="bold") +
  scale_color_identity() +
  geom_vline(xintercept=2009.5, linetype="dashed", linewidth=1.0, color="#0B0B0B") +
  # both labels sit above the panel rather than on top of the top band, which
  # needs clip="off" on the coord below and headroom in the top plot margin
  annotate("text", x=2009.5, y=1.02, label="Census coding change", hjust=0.5, vjust=0, size=3.1, fontface="bold", color="#0B0B0B") +
  annotate("text", x=2020, y=1.02, label="No data", hjust=0.5, vjust=0, size=3.1, fontface="bold", color="#0B0B0B") +
  scale_fill_manual(values=TASKAREA_COLORS, guide="none") +
  # no limits= on either scale: it drops rows outside the range rather than
  # zooming, and the stack tops land a hair above 1.0 in floating point, which
  # punched gaps in the top band. expand=c(0,0) sets the areas flush to both axes.
  scale_y_continuous(labels=function(v) sprintf("%.0f%%", v*100), breaks=seq(0, 1, 0.25),
                      expand=c(0, 0)) +
  scale_x_continuous(breaks=2008:2023, expand=c(0, 0)) +
  coord_cartesian(ylim=c(0, 1), xlim=c(2008, 2023), clip="off") +
  labs(title="Average County Task Group Value by Year",
       x="Year", y="Mean Group Value Across Counties") +
  # y tick labels lifted slightly so the 0% label clears the 2008 label in the
  # bottom-left corner; gridlines are off here, so nothing falls out of alignment
  theme(axis.text.x=element_text(angle=0, hjust=0.5),
        axis.text.y=element_text(vjust=0.15),
        panel.grid.major=element_blank(), panel.grid.minor=element_blank(),
        # right margin widened only enough for the 2023 tick label, which sat half
        # outside the plot. The panel itself is untouched, so the bands stay flush
        # to the axis with no whitespace added inside the plot area
        plot.margin=margin(26, 16, 5.5, 5.5))

Individual counties hold their positions too, beyond the national aggregate. <a href="#tbl-rankcorr" class="quarto-xref">Table 9</a> reports the rank correlation of each group’s county ordering between 2010 and 2023, computed from 2010 onward to step over the coding change. Routine manual and non-routine cognitive hold their orderings most strongly, since the counties with the most of each in 2010 are largely the same counties in 2023. Non-routine manual is more mobile. Routine cognitive reorders the most, which is consistent with it being the one group carrying substantial within county movement in the variance decomposition of <a href="#sec-analysis" class="quarto-xref">Section 4</a>.

In [85]:
from scipy.stats import spearmanr

wide_2010=df[df["year"]==2010].set_index("county_fips")
wide_2023=df[df["year"]==2023].set_index("county_fips")
common=wide_2010.index.intersection(wide_2023.index)

rank_rows=[]
for col, name in area_names.items():
    rho=spearmanr(wide_2010.loc[common, col], wide_2023.loc[common, col]).statistic
    rank_rows.append({"Task Group":name, "Rank Correlation":rho})
rankcorr_df=pd.DataFrame(rank_rows)

style_table(GT(rankcorr_df)
  .fmt_number(columns="Rank Correlation", decimals=2)
  .cols_align(align="center", columns="Rank Correlation")
  .cols_label(**{"Rank Correlation": html("Rank<br>Correlation")})
  .cols_width(cases={"Task Group": "140px", "Rank Correlation": "110px"})
  .tab_source_note("Computed from 2010 onward to avoid the Census occupation coding change."))

/opt/anaconda3/envs/capstone/lib/python3.12/site-packages/great_tables/_render_checks.py:37: RenderWarning: Rendering table with .cols_width() in Quarto may result in unexpected behavior. This is because Quarto performs custom table processing. Either use all percentage widths, or set .tab_options(quarto_disable_processing=True) to disable Quarto table processing.
  warnings.warn(

The stability is visible county by county, not only in the ordering, over the same two years used in <a href="#tbl-rankcorr" class="quarto-xref">Table 9</a> (<a href="#fig-taskchange-map" class="quarto-xref">Figure 27</a>). Routine manual and non-routine manual, the two groups with almost no net movement in <a href="#fig-taskarea" class="quarto-xref">Figure 26</a>, are centered near zero and run in both directions across counties. That mixed direction is consistent with variation sitting between counties rather than within them. Routine cognitive and non-routine cognitive, the pair whose national levels moved most, show a change that is both larger and far more uniform in direction, which matches the within county movement identified in the variance decomposition of <a href="#sec-analysis" class="quarto-xref">Section 4</a>.

In [86]:
import geopandas as gpd

change_wide=(wide_2023.loc[common, list(area_names.keys())]
             -wide_2010.loc[common, list(area_names.keys())])*100
change_wide.columns=[area_names[c] for c in change_wide.columns]
change_long=change_wide.reset_index().melt(id_vars="county_fips", var_name="group", value_name="change")
change_long["fips"]=change_long["county_fips"].astype(str).str.zfill(5)

gdf_change=gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip")
gdf_change=gdf_change.rename(columns={"GEOID":"fips"})
gdf_change=gdf_change[~gdf_change["STATEFP"].isin(["02","15","60","66","69","72","78"])]

# cross every continental county with all four groups so counties outside the panel
# still render, grey, in every facet, matching the coverage convention in fig-afford-map
groups_order=list(area_names.values())
county_group_grid=gdf_change[["fips","geometry"]].merge(
    pd.DataFrame({"group":groups_order}), how="cross")
merged_change=county_group_grid.merge(change_long[["fips","group","change"]],
                                      on=["fips","group"], how="left")
os.makedirs("output", exist_ok=True)
merged_change.to_file("output/taskchange_map.geojson", driver="GeoJSON")

change_bound=float(change_long["change"].abs().quantile(0.98))

In [87]:
%%R -i change_long -w 9 -h 5.5 -u in -r 150 -b transparent
change_long$group <- factor(change_long$group,
    levels=rev(c("Routine Cognitive","Routine Manual","Non-Routine Cognitive","Non-Routine Manual")))
change_med <- aggregate(change ~ group, change_long, median)
change_bound_strip <- max(abs(change_long$change), na.rm=TRUE) * 1.05

ggplot(change_long, aes(x=change, y=group, color=group)) +
  geom_vline(xintercept=0, linewidth=0.4, color="#0B0B0B") +
  geom_jitter(height=0.32, width=0, size=1.1, alpha=0.35) +
  geom_point(data=change_med, aes(x=change, y=group), inherit.aes=FALSE,
             shape="|", size=10, color="#0B0B0B", stroke=1.5) +
  scale_color_manual(values=TASK_COLORS, guide="none") +
  scale_x_continuous(limits=c(-change_bound_strip, change_bound_strip),
                      labels=function(v) sprintf("%+.0fpp", v)) +
  labs(title="County Task Group Change, 2010 to 2023",
       x="Percentage Point Change", y=NULL) +
  theme(panel.grid.major.y=element_blank())

Most counties moved toward non-routine cognitive work between 2010 and 2023, once task group change is collapsed onto the two axes that structure the four groups in <a href="#fig-taskframework" class="quarto-xref">Figure 1</a> (<a href="#fig-taskchange-arrows" class="quarto-xref">Figure 28</a>). That direction is consistent with routine cognitive and non-routine cognitive being the pair that moved most in <a href="#fig-taskarea" class="quarto-xref">Figure 26</a>. The counties colored red moved the other way, toward the three groups this study associates with lower purchasing power.

In [88]:
# shares the 2010 to 2023 window with tbl-rankcorr and fig-taskchange-map: it covers
# more counties than a 2008 start (a county needs data at both endpoints to have a
# change) and avoids the Census occupation coding change
manual_change_arrows=change_wide["Routine Manual"]+change_wide["Non-Routine Manual"]
nonroutine_change_arrows=change_wide["Non-Routine Cognitive"]+change_wide["Non-Routine Manual"]

arrow_shift_df=pd.DataFrame({
    "fips": change_wide.index.astype(str).str.zfill(5),
    "manual_change": manual_change_arrows.values,
    "nonroutine_change": nonroutine_change_arrows.values,
})
arrow_shift_df["shift_magnitude"]=np.sqrt(arrow_shift_df["manual_change"]**2+arrow_shift_df["nonroutine_change"]**2)

# risk direction relative to the regression's reference group: non-routine cognitive is the
# only one of the four task groups positively associated with purchasing power, so a county
# moving away from it (a negative change in its share) is moving toward the three groups that
# are all associated with lower purchasing power
arrow_shift_df["risk_direction"]=np.where(
    change_wide["Non-Routine Cognitive"].values<0,
    "Away From Non-Routine Cognitive", "Toward Non-Routine Cognitive")

# cap the top 5 percent of shifts so a handful of outlier counties don't dominate the figure;
# direction is unchanged, only the arrow length is scaled back for those counties
mag_cap=float(arrow_shift_df["shift_magnitude"].quantile(0.95))
scale_factor=np.where(arrow_shift_df["shift_magnitude"]>mag_cap,
                      mag_cap/arrow_shift_df["shift_magnitude"], 1.0)
arrow_shift_df["manual_change_capped"]=arrow_shift_df["manual_change"]*scale_factor
arrow_shift_df["nonroutine_change_capped"]=arrow_shift_df["nonroutine_change"]*scale_factor

gdf_arrows=gdf_change.to_crs(5070)
gdf_arrows["cx"]=gdf_arrows.geometry.centroid.x
gdf_arrows["cy"]=gdf_arrows.geometry.centroid.y

arrow_data=gdf_arrows[["fips","cx","cy"]].merge(arrow_shift_df, on="fips", how="inner")

In [89]:
%%R -i arrow_data -w 9.5 -h 6.5 -u in -r 150 -b transparent
suppressMessages(library(sf))

ARROW_SCALE <- 8000
arrow_data$xend <- arrow_data$cx + arrow_data$manual_change_capped * ARROW_SCALE
arrow_data$yend <- arrow_data$cy + arrow_data$nonroutine_change_capped * ARROW_SCALE
arrow_data$risk_direction <- factor(arrow_data$risk_direction,
    levels=c("Away From Non-Routine Cognitive","Toward Non-Routine Cognitive"))

counties_arrows <- st_read("output/taskchange_map.geojson", quiet=TRUE)
counties_arrows <- counties_arrows[!duplicated(counties_arrows$fips),]

ggplot() +
  geom_sf(data=counties_arrows, fill="#FCFCFB", color="#E1E0D9", linewidth=0.08) +
  geom_segment(data=arrow_data,
               aes(x=cx, y=cy, xend=xend, yend=yend, color=risk_direction, linewidth=shift_magnitude),
               arrow=arrow(length=unit(1.3,"mm"), type="closed"), lineend="round", alpha=0.75) +
  scale_linewidth_continuous(range=c(0.2, 1.0), guide="none") +
  # Orange and purple rather than red and green: these are two directions of one
  # shift, neither good nor bad, so a valenced pair would say the wrong thing.
  # Same reasoning as fig-between-within.
  scale_color_manual(values=c("Away From Non-Routine Cognitive"=ORANGE,
                              "Toward Non-Routine Cognitive"=PURPLE), name=NULL) +
  coord_sf(crs=st_crs(5070), datum=NA) +
  theme_void(base_size=11) +
  labs(title="Direction of County Task Group Shift, 2010 to 2023") +
  theme(legend.position="bottom",
        plot.title=element_text(face="bold", size=13, hjust=0.5),
        plot.background=element_rect(fill=NA, color=NA))

Finally, the coefficients themselves hold across time. <a href="#fig-stability" class="quarto-xref">Figure 29</a> refits the log panel regression on the pre pandemic window, 2010 to 2019, and the post pandemic window, 2021 to 2023, the same windows named in <a href="#sec-analysis" class="quarto-xref">Section 4</a>. Reporting these in percent rather than dollars keeps the three windows comparable, since nominal purchasing power rose substantially across the panel and a dollar coefficient in the later window is measured against a larger base. The task group ordering is identical in both windows and in the full panel, and the magnitudes move modestly. The association is therefore not an artifact of any one period, recession, recovery, or pandemic era.

In [90]:
def fit_window_log(frame):
    Xw=pd.concat([frame[res_cols],
                  pd.get_dummies(frame["year"], prefix="year", drop_first=True).astype(float)], axis=1)
    Xw=sm.add_constant(Xw)
    return sm.OLS(np.log(frame["affordability_salary"]), Xw).fit(
        cov_type="cluster", cov_kwds={"groups": frame["county_fips"]})

windows={"Full panel": reg,
         "2010 to 2019": reg[(reg["year"]>=2010)&(reg["year"]<=2019)],
         "2021 to 2023": reg[reg["year"]>=2021]}

stab_rows=[]
for term, name in zip(group_terms, group_names):
    row={"Task group":name}
    for wname, frame in windows.items():
        b=fit_window_log(frame).params[term]
        row[wname]=(np.exp(b*0.01)-1)*100
    stab_rows.append(row)
stability_df=pd.DataFrame(stab_rows)

stab_long=stability_df.melt(id_vars="Task group", var_name="window", value_name="value")
range_df=stability_df.assign(
    lo=stability_df[["Full panel","2010 to 2019","2021 to 2023"]].min(axis=1),
    hi=stability_df[["Full panel","2010 to 2019","2021 to 2023"]].max(axis=1),
)[["Task group","lo","hi"]]

In [91]:
%%R -i stab_long -i range_df -w 9 -h 4 -u in -r 150 -b transparent
BLACK <- "#252525"

group_order <- c("Routine Manual", "Routine Cognitive", "Non-Routine Manual")
stab_long$window <- factor(stab_long$window, levels=c("2010 to 2019", "Full panel", "2021 to 2023"))
stab_long$`Task group` <- factor(stab_long$`Task group`, levels=group_order)
range_df$`Task group` <- factor(range_df$`Task group`, levels=group_order)

window_colors <- c("2010 to 2019"=BLUE, "Full panel"=BLACK, "2021 to 2023"=ORANGE)
window_shapes <- c("2010 to 2019"=16, "Full panel"=18, "2021 to 2023"=16)
window_sizes  <- c("2010 to 2019"=3.2, "Full panel"=4.6, "2021 to 2023"=3.2)

# the three windows sit within 0.04 of each other for some groups, so stacking all
# three value labels at one height would overlap. Place them by rank instead:
# leftmost reads left, rightmost reads right, and the middle one sits above.
stab_long$rank <- ave(stab_long$value, stab_long$`Task group`, FUN=rank)
lab_pad <- diff(range(stab_long$value)) * 0.03

ggplot() +
  geom_vline(xintercept=0, linetype="dashed", linewidth=0.5, color=BLACK) +
  geom_segment(data=range_df, aes(x=lo, xend=hi, y=`Task group`, yend=`Task group`,
                                  linewidth="Range across windows"),
               color="#D6D6D2", lineend="round") +
  geom_point(data=stab_long, aes(x=value, y=`Task group`, color=window, shape=window, size=window)) +
  geom_text(data=subset(stab_long, rank==1),
            aes(x=value - lab_pad, y=`Task group`, label=sprintf("%.2f", value), color=window),
            hjust=1, size=3.1, fontface="bold", show.legend=FALSE) +
  geom_text(data=subset(stab_long, rank==3),
            aes(x=value + lab_pad, y=`Task group`, label=sprintf("%.2f", value), color=window),
            hjust=0, size=3.1, fontface="bold", show.legend=FALSE) +
  geom_text(data=subset(stab_long, rank==2),
            aes(x=value, y=`Task group`, label=sprintf("%.2f", value), color=window),
            vjust=-1.5, size=3.1, fontface="bold", show.legend=FALSE) +
  scale_color_manual(values=window_colors, name=NULL,
                     labels=c("2010 to 2019","Full panel (2008 to 2023)","2021 to 2023")) +
  scale_shape_manual(values=window_shapes, name=NULL,
                     labels=c("2010 to 2019","Full panel (2008 to 2023)","2021 to 2023")) +
  scale_size_manual(values=window_sizes, guide="none") +
  scale_linewidth_manual(values=c("Range across windows"=3.2), name=NULL) +
  scale_x_continuous(expand=expansion(mult=c(0.10, 0.06))) +
  labs(title="Coefficient Stability Across Time Windows",
       x="Percent Change per One Percentage Point Shift From Non-Routine Cognitive Work",
       y=NULL) +
  theme(legend.position="bottom",
        axis.text.y=element_text(face="bold", size=9.5),
        panel.grid.major.y=element_blank())

Taken together, the results show that this pattern is not limited to a single year or model. Counties with more routine and manual work tended to have lower purchasing power at the beginning of the study period, and that pattern remained largely unchanged fifteen years later. <a href="#sec-conclusions" class="quarto-xref">Section 6</a> discusses what these findings mean, their limits, and what questions remain.

# Conclusions

## Summary of findings

The introduction opened with national concern about automation, and at the county level the evidence assembled here shows that a county’s task groups are associated with differences in residents’ purchasing power. Across 840 counties and fifteen years, counties with higher concentrations of routine and manual work tend to have lower purchasing power than counties where non-routine cognitive work dominates. This pattern persists after accounting for poverty, unemployment, population, and year. At the panel mean, a one percentage point shift from non-routine cognitive to non-routine manual work is associated with roughly \$846 less in purchasing power, with a 95 percent interval of plus or minus \$38. The diagnostics suggest this relationship is better represented as proportional rather than linear in dollars. Neither the random forest nor the neural network improved on the log linear specification, since across five grouped cross validation folds the difference between the linear model and the forest was smaller than the variation between folds.

## Contributions

The introduction identified three gaps between the existing task based literature and a county level analysis of purchasing power. This study addresses each of them. First, prior work has typically evaluated labor market outcomes using wages, employment, or related measures that do not account for geographic differences in prices. Here, the outcome is median household income adjusted by the BEA Regional Price Parities, allowing the analysis to compare what income can purchase across counties rather than income alone. This distinction matters because two counties with similar incomes may provide very different standards of living once local prices are considered.

Second, much of the task based literature has relied on broader labor market units that group several counties together. This study instead uses the county as the unit of analysis, a finer scale that connects task groups more directly to the economic conditions experienced by local residents. The county year panel also allows those differences to be followed from 2008 through 2023 rather than treated as a single cross sectional comparison.

Third, the study does not assume that the relationship between task groups and purchasing power is adequately represented by a simple linear model. The functional form is evaluated directly through diagnostic testing, a logged specification, and comparison with more flexible machine learning models. The results favor a proportional interpretation of the association, while the random forest and neural network provide little improvement once that form is accounted for. This strengthens the case for retaining the more interpretable log linear model rather than adding complexity without a corresponding gain in performance. These extensions move the task framework from a description of occupational structure toward a county level measure that can be related directly to local purchasing power using publicly available and reproducible data.

The findings also have practical relevance for policymakers, since task groups provide information about local economic conditions that is not fully captured by poverty or unemployment alone. Counties with similar levels of conventional economic distress can still differ in the kinds of work their residents perform and in the purchasing power associated with that employment structure. The task measures offer an additional way to identify places where economic vulnerability may not be fully visible in standard indicators.

For economic development agencies and other organizations making place based investment decisions, the purchasing power measure adds a second distinction. Combining it with task groups helps identify where counties differ from what income alone suggests, and what features of the local labor market are associated with those differences. The broader contribution of the study is an additional lens on existing measures of economic distress. Task groups capture a relatively persistent feature of local economies, one that helps explain differences in purchasing power beyond poverty, unemployment, and population alone.

## Limitations

Five limitations bound what the results can support. The findings are associations rather than causal relationships, since counties were not assigned their task groups. Any factor tied to both the work a county contains and its purchasing power could account for part of the relationship.

Because the panel covers only counties above 65,000 residents, rural counties are underrepresented, and the estimates therefore describe the 84 percent of the population living in more populous counties rather than counties in general.

The panel regression carries no state indicators, and its estimates absorb whatever varies systematically by state. The predictive models, which do include state indicators, suggest that variation adds little once poverty, unemployment, and time are present, but the two specifications are not identical on this point.

Purchasing power is also not deflated to a constant base year, and the year indicators absorb national price drift alongside every other change common to a year, rather than the panel correcting for inflation directly. <a href="#sec-analysis" class="quarto-xref">Section 4</a> treats the year indicators as nuisance parameters for this reason, which means their coefficients cannot be read as a measure of the growth of purchasing power over time.

A majority of panel rows carry a state level price parity rather than a local one. The outcome is measured more coarsely in nonmetropolitan counties, and the analysis does not separate the two.

## Ethical considerations

The design also carries ethical boundaries that govern how the results should be read. Because the unit of analysis is the county, every claim describes places rather than people, and inferring anything about an individual worker from these results would be an ecological fallacy.

The counties missing from the panel are missing because of a publication threshold in the source data, not a choice made here. That is a representation bias the analysis inherits rather than introduces, and it means that applying these findings to those counties would be an extrapolation beyond the data.

Presenting these estimates as more than associations could misdirect policy, because it would point spending at a county’s task groups when the confounding factor named among the limitations above accounts for the difference instead. The estimates support describing where the gaps sit, not prescribing what to change.

Describing counties as exposed carries a related risk, since that language could stigmatize communities or steer investment away from the places these findings are meant to help.

The study uses public aggregate data only, and no individual’s information enters the analysis at any point.

## Future directions

Two directions follow from these limitations, the first of which is the coverage gap, which could plausibly be narrowed with satellite imagery. Jean et al. ([2016](#ref-Jean2016)) estimate local economic conditions from daytime and nighttime imagery where survey data are thin. Applied here, that approach could produce purchasing power estimates for the counties below the American Community Survey threshold, which would show whether the geography reported above extends to the places this panel cannot see. It would not extend the association itself, since occupational estimates remain unavailable for those counties. But confirming that the outcome pattern continues below the threshold would establish that the geography reported here is a feature of the country rather than of the sample.

The second direction is the harder one, since establishing why the two move together would require a source of variation in task groups that is unrelated to purchasing power, such as plant openings and closures or technology adoption shocks. That would call for a separate study, using different data and a design capable of isolating plausible exogenous changes in local task groups.

## Conclusion

This study shows that task groups are a measurable characteristic of counties, and that they carry information about what residents can afford. These are associations. The study does not establish why the two move together, since counties differ in far more than their task groups. The counties whose work was most routine and most manual entered the study period with less purchasing power, and fifteen years later they largely still have less. Task groups moved little across those years, though not uniformly. County orderings held tightly for routine manual and non-routine cognitive work, and much more loosely for routine cognitive. The gap they mark therefore looks like a persistent feature of these places rather than a temporary one. Whether it would narrow under any particular policy is a question this design cannot answer. The introduction opened with workers who expected automation to reach their own jobs. This study cannot say whether that expectation is correct. It does show that the counties already doing the most routine and manual work are the ones where a dollar of income goes least far. Anyone reading only the poverty rate is looking at an incomplete picture of where purchasing power in America is strained, and a county’s task groups are part of what completes that picture.

# References

## Software

Data processing, modeling, and machine learning were carried out in Python using pandas ([McKinney 2010](#ref-pandas2010)), NumPy ([Harris et al. 2020](#ref-numpy2020)), SciPy ([Virtanen et al. 2020](#ref-scipy2020)), statsmodels ([Seabold and Perktold 2010](#ref-statsmodels2010)), scikit-learn ([Pedregosa et al. 2011](#ref-sklearn2011)), TensorFlow ([Abadi et al. 2016](#ref-tensorflow2016)), and GeoPandas ([Jordahl et al. 2020](#ref-geopandas2020)). Figures were produced in R ([R Core Team 2024](#ref-rcoreteam2024)) using ggplot2 ([Wickham 2016](#ref-ggplot2_2016)), patchwork ([Pedersen 2024](#ref-patchwork2024)), scales ([Wickham, Pedersen, and Seidel 2023](#ref-scales2023)), and sf ([Pebesma 2018](#ref-sf2018)).

Abadi, Martín, Ashish Agarwal, Paul Barham, Eugene Brevdo, Zhifeng Chen, Craig Citro, Greg S. Corrado, et al. 2016. “TensorFlow: Large-Scale Machine Learning on Heterogeneous Distributed Systems.” <https://www.tensorflow.org/>.

Acemoglu, Daron, and David Autor. 2011. “Skills, Tasks and Technologies: Implications for Employment and Earnings.” In *Handbook of Labor Economics*, edited by David Card and Orley Ashenfelter, 4:1043–1171. Elsevier. <https://doi.org/10.1016/S0169-7218(11)02410-5>.

Autor, David H., and David Dorn. 2013. “The Growth of Low-Skill Service Jobs and the Polarization of the US Labor Market.” *American Economic Review* 103 (5): 1553–97. <https://doi.org/10.1257/aer.103.5.1553>.

Autor, David H., Frank Levy, and Richard J. Murnane. 2003. “The Skill Content of Recent Technological Change: An Empirical Exploration.” *The Quarterly Journal of Economics* 118 (4): 1279–1333. <https://doi.org/10.1162/003355303322552801>.

Chiripanhura, Blessing. 2011. “Median and Mean Income Analyses: Their Implications for Material Living Standards and National Well-Being.” *Economic & Labour Market Review* 5: 45–63. <https://doi.org/10.1057/elmr.2011.17>.

Harris, Charles R., K. Jarrod Millman, Stéfan J. van der Walt, Ralf Gommers, Pauli Virtanen, David Cournapeau, Eric Wieser, et al. 2020. “Array Programming with NumPy.” *Nature* 585 (7825): 357–62. <https://doi.org/10.1038/s41586-020-2649-2>.

Jean, Neal, Marshall Burke, Michael Xie, W. Matthew Davis, David B. Lobell, and Stefano Ermon. 2016. “Combining Satellite Imagery and Machine Learning to Predict Poverty.” *Science* 353 (6301): 790–94. <https://doi.org/10.1126/science.aaf7894>.

Jordahl, Kelsey, Joris Van den Bossche, Martin Fleischmann, Jacob Wasserman, James McBride, Jeffrey Gerard, Jeff Tratner, et al. 2020. “<span class="nocase">geopandas/geopandas</span>.” <https://doi.org/10.5281/zenodo.3946761>.

McKinney, Wes. 2010. “Data Structures for Statistical Computing in Python.” In *Proceedings of the 9th Python in Science Conference*, 56–61. <https://doi.org/10.25080/Majora-92bf1922-00a>.

Moretti, Enrico. 2013. “Real Wage Inequality.” *American Economic Journal: Applied Economics* 5 (1): 65–103. <https://doi.org/10.1257/app.5.1.65>.

Pebesma, Edzer. 2018. “Simple Features for R: Standardized Support for Spatial Vector Data.” *The R Journal* 10 (1): 439–46. <https://doi.org/10.32614/RJ-2018-009>.

Pedersen, Thomas Lin. 2024. *<span class="nocase">patchwork</span>: The Composer of Plots*. <https://CRAN.R-project.org/package=patchwork>.

Pedregosa, Fabian, Gaël Varoquaux, Alexandre Gramfort, Vincent Michel, Bertrand Thirion, Olivier Grisel, Mathieu Blondel, et al. 2011. “<span class="nocase">scikit-learn</span>: Machine Learning in Python.” *Journal of Machine Learning Research* 12: 2825–30. <https://jmlr.org/papers/v12/pedregosa11a.html>.

R Core Team. 2024. *R: A Language and Environment for Statistical Computing*. Vienna, Austria: R Foundation for Statistical Computing. <https://www.R-project.org/>.

Saad, Lydia. 2023. “More U.S. Workers Fear Technology Making Their Jobs Obsolete.” Gallup. <https://news.gallup.com/poll/510551/workers-fear-technology-making-jobs-obsolete.aspx>.

Seabold, Skipper, and Josef Perktold. 2010. “<span class="nocase">statsmodels</span>: Econometric and Statistical Modeling with Python.” In *Proceedings of the 9th Python in Science Conference*, 92–96. <https://doi.org/10.25080/Majora-92bf1922-011>.

Smith, Aaron, and Monica Anderson. 2017. “Automation in Everyday Life.” Pew Research Center. <https://www.pewresearch.org/internet/2017/10/04/automation-in-everyday-life/>.

U.S. Bureau of Economic Analysis. 2024. “Regional Price Parities by State and Metro Area.” <https://www.bea.gov/data/prices-inflation/regional-price-parities-state-and-metro-area>.

U.S. Bureau of Labor Statistics. 2024. “Local Area Unemployment Statistics.” <https://www.bls.gov/lau/>.

U.S. Census Bureau. 2023a. “Core Based Statistical Area Delineation Files.” <https://www.census.gov/geographies/reference-files/time-series/demo/metro-micro/delineation-files.html>.

U.S. Census Bureau. 2023b. “Vintage 2023 Population Estimates.” <https://www.census.gov/programs-surveys/popest.html>.

U.S. Census Bureau. 2024. “American Community Survey 1-Year Estimates.” <https://www.census.gov/programs-surveys/acs>.

Virtanen, Pauli, Ralf Gommers, Travis E. Oliphant, Matt Haberland, Tyler Reddy, David Cournapeau, Evgeni Burovski, et al. 2020. “SciPy 1.0: Fundamental Algorithms for Scientific Computing in Python.” *Nature Methods* 17 (3): 261–72. <https://doi.org/10.1038/s41592-019-0686-2>.

Wickham, Hadley. 2016. *<span class="nocase">ggplot2</span>: Elegant Graphics for Data Analysis*. Springer-Verlag New York. <https://ggplot2-book.org/>.

Wickham, Hadley, Thomas Lin Pedersen, and Dana Seidel. 2023. *<span class="nocase">scales</span>: Scale Functions for Visualization*. <https://CRAN.R-project.org/package=scales>.